# submission_lgb_only

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# A/Bテストフラグ（1つずつTrueにしてLBで評価）
# ============================================================
# まず全部Falseで元コードを再現 → LB 12.6を確認
# その後、1つずつTrueにして提出

TEST_A_WEIGHTED_KNN   = False   # 逆距離加重KNN追加
TEST_B_D2_KNN         = False   # d2空間でのKNN追加
TEST_C_KNN_DISTANCE   = False   # KNN距離を特徴量に追加
TEST_D_HIGHER_LGB_W   = False   # LGB重み 0.70 → 0.80
TEST_E_DROP_RIDGE     = False   # Ridge除外 (LGB 0.80 + PLS 0.20)
TEST_F_BAND_AREA      = False   # 水バンド面積1個追加

print("=" * 60)
print("📋 A/Bテスト設定:")
print(f"  A: 逆距離加重KNN  = {TEST_A_WEIGHTED_KNN}")
print(f"  B: d2空間KNN     = {TEST_B_D2_KNN}")
print(f"  C: KNN距離特徴量  = {TEST_C_KNN_DISTANCE}")
print(f"  D: LGB重み0.80   = {TEST_D_HIGHER_LGB_W}")
print(f"  E: Ridge除外     = {TEST_E_DROP_RIDGE}")
print(f"  F: バンド面積追加  = {TEST_F_BAND_AREA}")
print("=" * 60)

# ============================================================
# 1. データ読み込み（元コードと同一）
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

# 元コードの比率用
wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

# TEST_F用
band_water_5150 = np.where((wavenumbers >= 5000) & (wavenumbers <= 5300))[0]

# ============================================================
# 2. 前処理（元コードと同一）
# ============================================================

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 3. ブレンド重み
# ============================================================
if TEST_E_DROP_RIDGE:
    W_LGB, W_PLS, W_RDG = 0.80, 0.20, 0.00
elif TEST_D_HIGHER_LGB_W:
    W_LGB, W_PLS, W_RDG = 0.80, 0.10, 0.10
else:
    W_LGB, W_PLS, W_RDG = 0.70, 0.15, 0.15  # 元コードと同一

print(f"\n  重み: LGB={W_LGB}, PLS={W_PLS}, Ridge={W_RDG}")


# ============================================================
# 4. CVループ
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
final_pls = np.zeros(len(test))
final_rdg = np.zeros(len(test))

oof_lgb = np.zeros(len(train))
oof_pls = np.zeros(len(train))
oof_rdg = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── SNV（元コード同一）──
    snv_tr = apply_snv(X_tr_raw)
    snv_va = apply_snv(X_va_raw)
    snv_te = apply_snv(X_te_raw)

    # ── SG微分（元コード同一）──
    d1_tr = savgol_filter(snv_tr, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_va = savgol_filter(snv_va, window_length=15, polyorder=2, deriv=1, axis=1)
    d1_te = savgol_filter(snv_te, window_length=15, polyorder=2, deriv=1, axis=1)

    d2_tr = savgol_filter(snv_tr, window_length=11, polyorder=2, deriv=2, axis=1)
    d2_va = savgol_filter(snv_va, window_length=11, polyorder=2, deriv=2, axis=1)
    d2_te = savgol_filter(snv_te, window_length=11, polyorder=2, deriv=2, axis=1)

    # ── 元コードの特徴量 ──
    ratio_tr = (X_tr_raw[:, idx_1940] / (X_tr_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_va = (X_va_raw[:, idx_1940] / (X_va_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    ratio_te = (X_te_raw[:, idx_1940] / (X_te_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)

    std_tr = np.std(X_tr_raw, axis=1, keepdims=True)
    std_va = np.std(X_va_raw, axis=1, keepdims=True)
    std_te = np.std(X_te_raw, axis=1, keepdims=True)

    # ── PCA（元コード同一: 10次元）──
    pca = PCA(n_components=10, random_state=42)
    pca_tr = pca.fit_transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    # ── KNN（元コード同一: k=5, cosine）──
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_tr)

    # Train（自分除外: k=6→1つ目を捨てる）
    dist_tr, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    knn_ymean_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

    # Validation
    dist_va, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)

    # Test
    dist_te, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── TEST_A: 逆距離加重KNN ──
    if TEST_A_WEIGHTED_KNN:
        # Train（自分除外）
        w_tr = 1.0 / (dist_tr[:, 1:] + 1e-8)
        w_tr = w_tr / w_tr.sum(axis=1, keepdims=True)
        knn_weighted_tr = np.sum(w_tr * y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)

        w_va = 1.0 / (dist_va + 1e-8)
        w_va = w_va / w_va.sum(axis=1, keepdims=True)
        knn_weighted_va = np.sum(w_va * y_tr[ind_va], axis=1).reshape(-1, 1)

        w_te = 1.0 / (dist_te + 1e-8)
        w_te = w_te / w_te.sum(axis=1, keepdims=True)
        knn_weighted_te = np.sum(w_te * y_tr[ind_te], axis=1).reshape(-1, 1)

    # ── TEST_B: d2空間でのKNN ──
    if TEST_B_D2_KNN:
        pca_d2 = PCA(n_components=10, random_state=42)
        pca_d2_tr = pca_d2.fit_transform(d2_tr)
        pca_d2_va = pca_d2.transform(d2_va)
        pca_d2_te = pca_d2.transform(d2_te)

        knn_d2 = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn_d2.fit(pca_d2_tr)

        _, ind_d2_tr = knn_d2.kneighbors(pca_d2_tr, n_neighbors=6)
        knn_d2_tr = np.mean(y_tr[ind_d2_tr[:, 1:]], axis=1).reshape(-1, 1)

        _, ind_d2_va = knn_d2.kneighbors(pca_d2_va, n_neighbors=5)
        knn_d2_va = np.mean(y_tr[ind_d2_va], axis=1).reshape(-1, 1)

        _, ind_d2_te = knn_d2.kneighbors(pca_d2_te, n_neighbors=5)
        knn_d2_te = np.mean(y_tr[ind_d2_te], axis=1).reshape(-1, 1)

    # ── TEST_C: KNN距離 ──
    if TEST_C_KNN_DISTANCE:
        knn_dist_tr = np.mean(dist_tr[:, 1:], axis=1).reshape(-1, 1)
        knn_dist_va = np.mean(dist_va, axis=1).reshape(-1, 1)
        knn_dist_te = np.mean(dist_te, axis=1).reshape(-1, 1)

    # ── TEST_F: バンド面積 ──
    if TEST_F_BAND_AREA:
        area_tr = np.trapezoid(np.abs(d2_tr[:, band_water_5150]), axis=1).reshape(-1, 1)
        area_va = np.trapezoid(np.abs(d2_va[:, band_water_5150]), axis=1).reshape(-1, 1)
        area_te = np.trapezoid(np.abs(d2_te[:, band_water_5150]), axis=1).reshape(-1, 1)

    # ──────────────────────────────────────
    # LGB入力の組み立て
    # ──────────────────────────────────────
    lgb_parts_tr = [snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr]
    lgb_parts_va = [snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va]
    lgb_parts_te = [snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te]

    if TEST_A_WEIGHTED_KNN:
        lgb_parts_tr.append(knn_weighted_tr)
        lgb_parts_va.append(knn_weighted_va)
        lgb_parts_te.append(knn_weighted_te)

    if TEST_B_D2_KNN:
        lgb_parts_tr.append(knn_d2_tr)
        lgb_parts_va.append(knn_d2_va)
        lgb_parts_te.append(knn_d2_te)

    if TEST_C_KNN_DISTANCE:
        lgb_parts_tr.append(knn_dist_tr)
        lgb_parts_va.append(knn_dist_va)
        lgb_parts_te.append(knn_dist_te)

    if TEST_F_BAND_AREA:
        lgb_parts_tr.append(area_tr)
        lgb_parts_va.append(area_va)
        lgb_parts_te.append(area_te)

    feat_tr_lgb = np.hstack(lgb_parts_tr)
    feat_va_lgb = np.hstack(lgb_parts_va)
    feat_te_lgb = np.hstack(lgb_parts_te)

    if fold == 0:
        print(f"\n  📐 LGB入力次元: {feat_tr_lgb.shape[1]}")
        active_tests = [name for name, flag in [
            ('A:加重KNN', TEST_A_WEIGHTED_KNN),
            ('B:d2-KNN', TEST_B_D2_KNN),
            ('C:KNN距離', TEST_C_KNN_DISTANCE),
            ('F:バンド面積', TEST_F_BAND_AREA),
        ] if flag]
        if active_tests:
            print(f"     追加特徴量: {', '.join(active_tests)}")
        else:
            print(f"     追加特徴量: なし（元コード完全再現）")

    # ─── モデル1: LightGBM（元コードと完全同一パラメータ）───
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr_lgb, y_tr,
        eval_set=[(feat_va_lgb, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va_lgb = np.expm1(lgb_model.predict(feat_va_lgb))
    p_te_lgb = np.expm1(lgb_model.predict(feat_te_lgb))

    # ─── モデル2: PLS（元コード同一: nc=7）───
    pls_model = PLSRegression(n_components=7)
    pls_model.fit(d2_tr, y_tr)
    p_va_pls = np.expm1(pls_model.predict(d2_va).flatten())
    p_te_pls = np.expm1(pls_model.predict(d2_te).flatten())

    # ─── モデル3: Ridge（元コード同一: scalerなし）───
    feat_tr_rdg = np.hstack([pca_tr, knn_ymean_tr, ratio_tr, std_tr])
    feat_va_rdg = np.hstack([pca_va, knn_ymean_va, ratio_va, std_va])
    feat_te_rdg = np.hstack([pca_te, knn_ymean_te, ratio_te, std_te])

    rdg_model = Ridge(alpha=10.0, random_state=42)
    rdg_model.fit(feat_tr_rdg, y_tr)
    p_va_rdg = np.expm1(rdg_model.predict(feat_va_rdg))
    p_te_rdg = np.expm1(rdg_model.predict(feat_te_rdg))

    # ─── ブレンド ───
    p_va_blend = p_va_lgb * W_LGB + p_va_pls * W_PLS + p_va_rdg * W_RDG

    oof_lgb[va_idx] = p_va_lgb
    oof_pls[va_idx] = p_va_pls
    oof_rdg[va_idx] = p_va_rdg

    final_lgb += p_te_lgb / 5
    final_pls += p_te_pls / 5
    final_rdg += p_te_rdg / 5

    y_va_real = np.expm1(y_va)
    rmse_lgb = np.sqrt(mean_squared_error(y_va_real, p_va_lgb))
    rmse_pls = np.sqrt(mean_squared_error(y_va_real, p_va_pls))
    rmse_rdg = np.sqrt(mean_squared_error(y_va_real, p_va_rdg))
    rmse_blend = np.sqrt(mean_squared_error(y_va_real, p_va_blend))
    fold_rmses.append(rmse_blend)

    print(f"  LGB   : {rmse_lgb:.4f}")
    print(f"  PLS   : {rmse_pls:.4f}")
    print(f"  Ridge : {rmse_rdg:.4f}")
    print(f"  🌟Blend: {rmse_blend:.4f}")


# ============================================================
# 5. 評価 & 提出
# ============================================================
print(f"\n{'='*60}")
y_true_real = np.expm1(y_train_log)

# OOF
oof_blend = oof_lgb * W_LGB + oof_pls * W_PLS + oof_rdg * W_RDG
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_blend))

print(f"📊 LGB      OOF: {np.sqrt(mean_squared_error(y_true_real, oof_lgb)):.4f}")
print(f"📊 PLS      OOF: {np.sqrt(mean_squared_error(y_true_real, oof_pls)):.4f}")
print(f"📊 Ridge    OOF: {np.sqrt(mean_squared_error(y_true_real, oof_rdg)):.4f}")
print(f"🌟 Blend    OOF: {oof_rmse:.4f}")
print(f"📊 Fold平均:     {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# 重み感度
print(f"\n  --- 重み感度 ---")
for name, (wl, wp, wr) in [
    ("0.70/0.15/0.15 (元)", (0.70, 0.15, 0.15)),
    ("0.75/0.15/0.10",     (0.75, 0.15, 0.10)),
    ("0.80/0.10/0.10",     (0.80, 0.10, 0.10)),
    ("0.80/0.20/0.00",     (0.80, 0.20, 0.00)),
    ("0.60/0.25/0.15",     (0.60, 0.25, 0.15)),
    ("LGB単独",            (1.00, 0.00, 0.00)),
    ("PLS単独",            (0.00, 1.00, 0.00)),
]:
    b = oof_lgb * wl + oof_pls * wp + oof_rdg * wr
    r = np.sqrt(mean_squared_error(y_true_real, b))
    print(f"    {name:25s} OOF: {r:.4f}")

# 樹種別
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'Blend':>7s} {'LGB':>7s} {'PLS':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    b_s = oof_blend[mask.values]
    l_s = oof_lgb[mask.values]
    p_s = oof_pls[mask.values]
    rmse_b = np.sqrt(np.mean((y_s - b_s)**2))
    rmse_l = np.sqrt(np.mean((y_s - l_s)**2))
    rmse_p = np.sqrt(np.mean((y_s - p_s)**2))
    bias = np.mean(y_s - b_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_b:7.2f} {rmse_l:7.2f} {rmse_p:7.2f} {bias:+7.2f}")

# 提出ファイル
final_blend = final_lgb * W_LGB + final_pls * W_PLS + final_rdg * W_RDG
final_blend = np.clip(final_blend, 0, None)

submit[1] = final_blend
out = 'submission_ab_test.csv'
submit.to_csv(out, index=False, header=False)
print(f"\n✅ 提出: {out}")
print(f"📈 min={final_blend.min():.1f}%, median={np.median(final_blend):.1f}%, "
      f"max={final_blend.max():.1f}%")

# 個別モデル提出
for name, preds in [('lgb_only', final_lgb), ('pls_only', final_pls)]:
    s = submit.copy()
    s[1] = np.clip(preds, 0, None)
    fname = f'submission_{name}.csv'
    s.to_csv(fname, index=False, header=False)
    print(f"  📦 {fname}")

# submission_Mixup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# テスト選択
# ============================================================
TEST_MIXUP           = True    # Mixup augmentation
TEST_NOISE           = False   # ノイズ注入のみ
TEST_MIXUP_AND_NOISE = False   # 両方

if TEST_MIXUP_AND_NOISE:
    pattern_name = "Mixup + Noise"
elif TEST_MIXUP:
    pattern_name = "Mixup"
elif TEST_NOISE:
    pattern_name = "Noise注入"
else:
    pattern_name = "元コード再現"

use_mixup = TEST_MIXUP or TEST_MIXUP_AND_NOISE
use_noise = TEST_NOISE or TEST_MIXUP_AND_NOISE

print("=" * 60)
print(f"🧪 テスト: {pattern_name}")
if use_mixup:
    print("  → 異なる樹種のスペクトルを混合して疑似データ生成")
if use_noise:
    print("  → 小さなガウスノイズで測定ばらつきを模擬")
print("=" * 60)

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
            if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 2. Augmentation関数
# ============================================================

def mixup_augmentation(X, y, species, n_augment=500, alpha=0.3, seed=42):
    """
    異なる樹種のサンプルを混合して疑似データを生成
    
    X: 生スペクトル (n_samples, n_wavelengths)
    y: log1p(含水率)
    species: 樹種番号
    n_augment: 生成するサンプル数
    alpha: Beta分布のパラメータ（小さいほど元データに近い混合）
    """
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    
    X_aug = []
    y_aug = []
    
    for _ in range(n_augment):
        # 異なる樹種から1サンプルずつ選ぶ
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        
        # 混合比率（Beta分布）
        lam = rng.beta(alpha, alpha)
        
        # スペクトルと含水率を線形補間
        x_mix = lam * X[idx1] + (1 - lam) * X[idx2]
        y_mix = lam * y[idx1] + (1 - lam) * y[idx2]
        
        X_aug.append(x_mix)
        y_aug.append(y_mix)
    
    return np.array(X_aug), np.array(y_aug)


def noise_augmentation(X, y, n_augment=300, noise_std=0.002, seed=42):
    """
    元データに小さなノイズを加えて疑似データを生成
    測定時のノイズ・プローブ距離変動を模擬
    """
    rng = np.random.RandomState(seed)
    
    indices = rng.choice(len(X), size=n_augment, replace=True)
    X_aug = X[indices] + rng.normal(0, noise_std, (n_augment, X.shape[1]))
    y_aug = y[indices]
    
    return X_aug, y_aug


# ============================================================
# 3. 特徴量作成関数（元コードと同一の処理をまとめる）
# ============================================================

def make_features(X_raw):
    """生スペクトルから元コードと同一の特徴量を作成"""
    snv = apply_snv(X_raw)
    d1 = savgol_filter(snv, window_length=15, polyorder=2, deriv=1, axis=1)
    ratio = (X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    std = np.std(X_raw, axis=1, keepdims=True)
    return snv, d1, ratio, std


# ============================================================
# 4. CVループ
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
oof_lgb = np.zeros(len(train))
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    tr_species_nums = groups.iloc[tr_idx].values
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train:{len(tr_idx)}, valid:{len(va_idx)})")
    print(f"   検証樹種: {list(va_species)}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── Augmentation（生スペクトルレベルで実施）──
    X_tr_aug = X_tr_raw.copy()
    y_tr_aug = y_tr.copy()

    if use_mixup:
        X_mix, y_mix = mixup_augmentation(
            X_tr_raw, y_tr, tr_species_nums,
            n_augment=500, alpha=0.3, seed=42+fold
        )
        X_tr_aug = np.vstack([X_tr_aug, X_mix])
        y_tr_aug = np.concatenate([y_tr_aug, y_mix])

    if use_noise:
        X_noi, y_noi = noise_augmentation(
            X_tr_raw, y_tr,
            n_augment=300, noise_std=0.002, seed=42+fold
        )
        X_tr_aug = np.vstack([X_tr_aug, X_noi])
        y_tr_aug = np.concatenate([y_tr_aug, y_noi])

    if fold == 0:
        print(f"   元データ: {len(X_tr_raw)} samples")
        print(f"   拡張後:   {len(X_tr_aug)} samples (+{len(X_tr_aug)-len(X_tr_raw)})")

    # ── 特徴量作成（元コードと同一処理）──
    snv_tr, d1_tr, ratio_tr, std_tr = make_features(X_tr_aug)
    snv_va, d1_va, ratio_va, std_va = make_features(X_va_raw)
    snv_te, d1_te, ratio_te, std_te = make_features(X_te_raw)

    # ── PCA（元の訓練データのみでfit → 拡張データにtransform）──
    snv_tr_orig = apply_snv(X_tr_raw)  # PCA fitは元データのみ
    pca = PCA(n_components=10, random_state=42)
    pca.fit(snv_tr_orig)  # 元データのみでfit
    pca_tr = pca.transform(snv_tr)
    pca_va = pca.transform(snv_va)
    pca_te = pca.transform(snv_te)

    # ── KNN（元データのみでfit）──
    pca_tr_orig = pca.transform(snv_tr_orig)
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(pca_tr_orig)

    # Train augmented
    _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
    # 元データ部分は自分除外、拡張部分はそのまま
    knn_ymean_tr = np.zeros(len(X_tr_aug))
    n_orig = len(X_tr_raw)
    for i in range(len(X_tr_aug)):
        neighbors = ind_tr[i]
        if i < n_orig:
            # 元データ: 自分を除外
            valid = neighbors[neighbors != i][:5]
        else:
            # 拡張データ: そのまま上位5つ
            valid = neighbors[:5]
        knn_ymean_tr[i] = np.mean(y_tr[:n_orig][valid])  # 元データのyのみ使用
    knn_ymean_tr = knn_ymean_tr.reshape(-1, 1)

    # Validation
    _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
    knn_ymean_va = np.mean(y_tr[:n_orig][ind_va], axis=1).reshape(-1, 1)

    # Test
    _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
    knn_ymean_te = np.mean(y_tr[:n_orig][ind_te], axis=1).reshape(-1, 1)

    # ── LGB入力（元コードと同一構成）──
    feat_tr = np.hstack([snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
    feat_va = np.hstack([snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
    feat_te = np.hstack([snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te])

    if fold == 0:
        print(f"   📐 LGB入力次元: {feat_tr.shape[1]}")

    # ── LightGBM（元コードと同一パラメータ）──
    lgb_model = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=5, num_leaves=31,
        subsample=0.8, colsample_bytree=0.3, random_state=42, verbosity=-1
    )
    lgb_model.fit(
        feat_tr, y_tr_aug,
        eval_set=[(feat_va, y_va)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    p_va = np.expm1(lgb_model.predict(feat_va))
    p_te = np.expm1(lgb_model.predict(feat_te))

    oof_lgb[va_idx] = p_va
    final_lgb += p_te / 5

    y_va_real = np.expm1(y_va)
    rmse = np.sqrt(mean_squared_error(y_va_real, p_va))
    fold_rmses.append(rmse)
    print(f"  🌟 LGB RMSE: {rmse:.4f}")

    # 特徴量重要度（Fold 0）
    if fold == 0:
        imp = lgb_model.feature_importances_
        n_snv = snv_tr.shape[1]
        n_d1 = d1_tr.shape[1]
        cat = {}
        pos = 0
        cat['SNV'] = np.sum(imp[pos:pos+n_snv]); pos += n_snv
        cat['d1'] = np.sum(imp[pos:pos+n_d1]); pos += n_d1
        cat['PCA'] = np.sum(imp[pos:pos+10]); pos += 10
        cat['KNN_mean'] = imp[pos]; pos += 1
        cat['ratio'] = imp[pos]; pos += 1
        cat['std'] = imp[pos]; pos += 1

        total = sum(cat.values())
        print(f"\n  📊 特徴量重要度:")
        for name, val in sorted(cat.items(), key=lambda x: -x[1]):
            pct = val / total * 100
            bar = '█' * int(pct / 2)
            print(f"     {name:15s}: {val:6.0f} ({pct:5.1f}%) {bar}")


# ============================================================
# 5. 全体評価
# ============================================================
print(f"\n{'='*60}")
print("📊 全体評価")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_lgb))

print(f"  🌟 LGB OOF RMSE: {oof_rmse:.4f}")
print(f"  📊 Fold平均 RMSE: {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")

# 樹種別
print(f"\n  --- 樹種別残差 ---")
print(f"  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_lgb[mask.values]
    rmse_s = np.sqrt(np.mean((y_s - p_s)**2))
    bias_s = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {rmse_s:7.2f} {bias_s:+7.2f}")


# ============================================================
# 6. 提出ファイル
# ============================================================
final_out = np.clip(final_lgb, 0, None)
submit[1] = final_out
out = f'submission_{pattern_name.replace(" ", "_")}.csv'
submit.to_csv(out, index=False, header=False)

print(f"\n✅ 提出ファイル: {out}")
print(f"📈 min={final_out.min():.1f}%, median={np.median(final_out):.1f}%, "
    f"max={final_out.max():.1f}%")

print(f"\n📌 全スコア比較:")
print(f"   LGB単独(元):       LB = 12.615 ← 現BEST")
print(f"   元Blend:           LB = 12.647")
print(f"   正則化強化:         LB = 12.760")
print(f"   Huber Loss:        LB = 12.770")
print(f"   PLS特徴量:          LB = 12.800")
print(f"   LGB+加重KNN:       LB = 12.940")
print(f"   LGB+d2:            LB = 13.410")
print(f"   今回({pattern_name}): LB = ???")

# ============================================================
# 7. Augmentationパラメータ感度分析（参考）
# ============================================================
print(f"\n{'='*60}")
print("📊 参考: Augmentationパラメータ候補")
print(f"{'='*60}")
print("""
もし今回が改善した場合、次に試すパラメータ:

n_augment:  300, 500, 1000  (多いほどデータ多様性↑、ノイズも↑)
alpha:      0.1, 0.3, 0.5   (小さいほど元データに近い混合)
noise_std:  0.001, 0.002, 0.005

もし悪化した場合:
→ alpha を小さく (0.1) → 元データからあまり離れない混合
→ n_augment を減らす (200) → 元データの比率を維持
""")

# submission_lgb35_x_bestpls_w075

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",          17.21, 12.615),
    ("元Blend(LGB/PLS/Ridge)",    14.10, 12.647),
    ("正則化強化(LGB単独)",        17.60, 12.760),
    ("Huber Loss",                17.53, 12.770),
    ("PLS予測を特徴量追加",        15.65, 12.800),
    ("逆距離加重KNN",             17.13, 12.940),
    ("d2(二次微分)追加",           16.12, 13.410),
    ("物理特徴量53個追加",         12.68, 14.500),
    ("Mixup seed42 α=0.3",       18.04, 11.800),
    ("Multi5+2w2000+3w1000",     None,  12.249),
    ("SafeMulti3(ensemble)",     17.64, 11.870),
    ("B_alpha05",                17.82, 12.326),
    ("E_water_bands_only",       19.43, 12.656),
    ("G2_alpha015_n500",         18.43, 11.935),
    ("seed35 LGB",               16.79, 11.644),
    ("C_plsonly2 seed42",        19.60, 11.788),
]

print("=" * 60)
print("📊 最新の勝利分析")
print("=" * 60)
print(f"""
  ★ seed=35: LB=11.644 (OOF=16.79) ← NEW BEST
  ★ C_plsonly2: LB=11.788 (OOF=19.60)
  
  重要発見:
  1. seed35はOOF16.79と「低い」のにLB最良
     → OOFスイートスポット理論は修正が必要
     → seed選択がLBに最大の影響を与える
  
  2. PLS 2成分(6次元)でLB 11.788
     → 含水率の本質は2変数で記述できる
     → 過学習のリスクがほぼゼロ
  
  戦略: seed35近傍の精密探索 + seed35×PLSブレンド
""")

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))

X_train_raw = train[spec_cols].values
X_test_raw  = test[spec_cols].values


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


def mixup_augmentation(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)


# ============================================================
# 2. LGBパイプライン（11.644再現用）
# ============================================================
def run_lgb_seed(seed, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_aug = apply_snv(X_aug)
        d1_aug  = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        snv_va  = apply_snv(X_va)
        d1_va   = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        snv_te  = apply_snv(X_test_raw)
        d1_te   = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va  = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te  = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va  = np.std(X_va, axis=1, keepdims=True)
        s_te  = np.std(X_test_raw, axis=1, keepdims=True)

        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va  = pca.transform(snv_va)
        pc_te  = pca.transform(snv_te)

        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pc_orig)

        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(pc_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it = knn.kneighbors(pc_te, 5)
        knn_te = np.mean(y_tr[it], axis=1).reshape(-1, 1)

        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, r_va, s_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_pred))
    if verbose:
        print(f"  seed={seed:4d}  OOF={oof_rmse:.4f}  "
              f"Fold={np.mean(fold_rmses):.4f}±{np.std(fold_rmses):.4f}")
    return {
        'seed': seed, 'oof': oof_rmse,
        'fold_mean': np.mean(fold_rmses),
        'test_pred': final_pred, 'oof_pred': oof_pred
    }


# ============================================================
# 3. PLS-onlyパイプライン（11.788再現用）
# ============================================================
def run_pls_only(n_comp=2, seed=42, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred   = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_augmentation(
            X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_orig = apply_snv(X_tr)
        snv_aug  = apply_snv(X_aug)
        snv_va   = apply_snv(X_va)
        snv_te   = apply_snv(X_test_raw)

        pls = PLSRegression(n_components=n_comp, scale=False)
        pls.fit(snv_orig, y_tr)

        ps_aug = pls.transform(snv_aug)
        ps_va  = pls.transform(snv_va)
        ps_te  = pls.transform(snv_te)
        pp_aug = pls.predict(snv_aug).ravel().reshape(-1,1)
        pp_va  = pls.predict(snv_va).ravel().reshape(-1,1)
        pp_te  = pls.predict(snv_te).ravel().reshape(-1,1)

        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(ps_orig)

        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(ps_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it_ = knn.kneighbors(ps_te, 5)
        knn_te = np.mean(y_tr[it_], axis=1).reshape(-1, 1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va  = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te  = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va  = np.std(X_va, axis=1, keepdims=True)
        s_te  = np.std(X_test_raw, axis=1, keepdims=True)

        ft = np.hstack([ps_aug, pp_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([ps_va, pp_va, knn_va, r_va, s_va])
        fe = np.hstack([ps_te, pp_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=4, num_leaves=15,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_pred))
    if verbose:
        print(f"  PLS{n_comp} seed={seed:4d}  OOF={oof_rmse:.4f}  "
              f"Fold={np.mean(fold_rmses):.4f}")
    return {
        'name': f'PLS{n_comp}_seed{seed}', 'oof': oof_rmse,
        'fold_mean': np.mean(fold_rmses),
        'test_pred': final_pred, 'oof_pred': oof_pred
    }


# ============================================================
# 4. 実験A: seed=35近傍の精密探索
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験A: seed=35近傍の精密探索")
print(f"{'='*60}")

fine_seeds = list(range(25, 40))  # 25~39の15個
lgb_results = {}

for s in fine_seeds:
    r = run_lgb_seed(s)
    lgb_results[s] = r

# 既知のseed35, 42も保持
r35 = run_lgb_seed(35)
lgb_results[35] = r35
r42 = run_lgb_seed(42)
lgb_results[42] = r42

# ソート
sorted_lgb = sorted(lgb_results.values(), key=lambda x: x['oof'])
print(f"\n  --- seed=25~42 精密探索結果（OOF順）---")
print(f"  {'seed':>6s} {'OOF':>8s} {'Fold平均':>8s}")
print(f"  {'─'*6} {'─'*8} {'─'*8}")
for r in sorted_lgb:
    m = " ← LB=11.644" if r['seed'] == 35 else (
        " ← LB=11.80" if r['seed'] == 42 else "")
    print(f"  {r['seed']:>6d} {r['oof']:>8.4f} "
          f"{r['fold_mean']:>8.4f}{m}")


# ============================================================
# 5. 実験B: PLS-onlyのseed探索
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験B: PLS-only (comp=2) のseed探索")
print(f"{'='*60}")

pls_seeds = [25, 30, 33, 34, 35, 36, 37, 38, 42, 0, 7, 77]
pls_results = {}

for s in pls_seeds:
    r = run_pls_only(n_comp=2, seed=s)
    pls_results[s] = r

sorted_pls = sorted(pls_results.values(), key=lambda x: x['oof'])
print(f"\n  --- PLS2 seed探索結果（OOF順）---")
print(f"  {'seed':>6s} {'OOF':>8s} {'Fold平均':>8s}")
for r in sorted_pls:
    m = " ← LB=11.788" if r['oof'] > 19.5 and 'seed42' in r['name'] else ""
    print(f"  {r['name']:<20s} {r['oof']:>8.4f} {r['fold_mean']:>8.4f}")


# ============================================================
# 6. 実験C: seed35 LGB × PLS-only ブレンド探索
# ============================================================
print(f"\n{'='*60}")
print("🔬 実験C: seed35 LGB × PLS-only ブレンド")
print("   2つの勝者を組み合わせる")
print(f"{'='*60}")

y_true = np.expm1(y_train_log)

# seed35のLGB結果
lgb_35 = lgb_results[35]

# 全PLS結果とのブレンド
blend_results = []

for pls_seed, pls_r in pls_results.items():
    for w in np.arange(0.5, 0.95, 0.05):
        oof_bl = w * lgb_35['oof_pred'] + (1-w) * pls_r['oof_pred']
        rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
        pred_bl = w * lgb_35['test_pred'] + (1-w) * pls_r['test_pred']
        blend_results.append({
            'name': f"LGB35×{w:.2f}+PLS2_s{pls_seed}×{1-w:.2f}",
            'lgb_seed': 35, 'pls_seed': pls_seed,
            'w': w, 'oof': rmse_bl,
            'test_pred': pred_bl,
            'oof_pred': oof_bl,
        })

# seed42のLGBも
lgb_42 = lgb_results[42]
for pls_seed, pls_r in pls_results.items():
    for w in np.arange(0.5, 0.95, 0.05):
        oof_bl = w * lgb_42['oof_pred'] + (1-w) * pls_r['oof_pred']
        rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
        pred_bl = w * lgb_42['test_pred'] + (1-w) * pls_r['test_pred']
        blend_results.append({
            'name': f"LGB42×{w:.2f}+PLS2_s{pls_seed}×{1-w:.2f}",
            'lgb_seed': 42, 'pls_seed': pls_seed,
            'w': w, 'oof': rmse_bl,
            'test_pred': pred_bl,
            'oof_pred': oof_bl,
        })

blend_results.sort(key=lambda x: x['oof'])

print(f"\n  Top 15 blends:")
print(f"  {'Name':<42s} {'OOF':>8s}")
print(f"  {'─'*42} {'─'*8}")
for b in blend_results[:15]:
    print(f"  {b['name']:<42s} {b['oof']:>8.4f}")

# 多様な組み合わせのTop5を選ぶ（同じseed組み合わせの重複排除）
seen_combos = set()
diverse_top = []
for b in blend_results:
    combo = (b['lgb_seed'], b['pls_seed'])
    if combo not in seen_combos:
        seen_combos.add(combo)
        diverse_top.append(b)
    if len(diverse_top) >= 5:
        break

print(f"\n  多様なTop5（seed組み合わせ重複排除）:")
for b in diverse_top:
    print(f"  {b['name']:<42s} OOF={b['oof']:.4f}")


# ============================================================
# 7. 提出ファイル作成
# ============================================================
print(f"\n{'='*60}")
print("📁 提出ファイル作成")
print(f"{'='*60}")

all_subs = {}

# Top3 LGB seeds（35近傍）
for r in sorted_lgb[:3]:
    out = submit_template.copy()
    out[1] = np.clip(r['test_pred'], 0, None)
    fname = f"submission_lgb_seed{r['seed']}.csv"
    out.to_csv(fname, index=False, header=False)
    all_subs[fname] = r['oof']
    print(f"  ✅ {fname} (OOF={r['oof']:.4f})")

# Best PLS seeds
for r in sorted_pls[:3]:
    out = submit_template.copy()
    out[1] = np.clip(r['test_pred'], 0, None)
    fname = f"submission_{r['name']}.csv"
    out.to_csv(fname, index=False, header=False)
    all_subs[fname] = r['oof']
    print(f"  ✅ {fname} (OOF={r['oof']:.4f})")

# Top5 blends (多様)
for b in diverse_top:
    out = submit_template.copy()
    out[1] = np.clip(b['test_pred'], 0, None)
    safe = b['name'].replace("×", "x").replace("+", "_")
    fname = f"submission_{safe}.csv"
    out.to_csv(fname, index=False, header=False)
    all_subs[fname] = b['oof']
    print(f"  ✅ {fname} (OOF={b['oof']:.4f})")

# seed35 × best_pls の特別ブレンド (w=0.7, 0.8)
best_pls_r = sorted_pls[0]
for w in [0.70, 0.75, 0.80]:
    pred = w * lgb_35['test_pred'] + (1-w) * best_pls_r['test_pred']
    oof  = w * lgb_35['oof_pred'] + (1-w) * best_pls_r['oof_pred']
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof))
    out = submit_template.copy()
    out[1] = np.clip(pred, 0, None)
    fname = f"submission_lgb35_x_bestpls_w{w:.2f}.csv"
    out.to_csv(fname, index=False, header=False)
    all_subs[fname] = oof_rmse
    print(f"  ✅ {fname} (OOF={oof_rmse:.4f})")


# ============================================================
# 8. 全スコア比較
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較")
print(f"{'='*60}")
print(f"  {'手法':<40s} {'OOF':>8s} {'LB':>8s}")
print(f"  {'─'*40} {'─'*8} {'─'*8}")

for name, oof, lb in HISTORY:
    oof_s = f"{oof:.2f}" if oof is not None else "---"
    m = " ★" if lb == 11.644 else (" ☆" if lb == 11.788 else (
        " ↓" if lb > 12.0 else ""))
    print(f"  {name:<40s} {oof_s:>8s} {lb:.3f}{m}")

print(f"  {'─'*40} {'─'*8} {'─'*8}")
print(f"  {'--- 今回の候補 ---':<40s}")

for fname, oof in sorted(all_subs.items(), key=lambda x: x[1]):
    short = fname.replace("submission_", "").replace(".csv", "")
    print(f"  {short:<40s} {oof:>8.2f} {'???':>8s}")


# ============================================================
# 9. 提出判断
# ============================================================
print(f"\n{'='*60}")
print("🧭 提出判断ガイド")
print(f"{'='*60}")

print(f"""
  ■ 確定した事実:
    - seed=35 LGB: LB=11.644 ★BEST
    - seed=42 LGB: LB=11.80
    - PLS2 seed42: LB=11.788
    - seed差(35 vs 42): LB差 0.156 (巨大)
  
  ■ 仮説: 
    OOFが低いseedほどLBが良い傾向がある
    (seed35: OOF=16.79→LB=11.644, seed42: OOF=18.04→LB=11.80)
    ただしサンプル2点なので確実ではない
  
  ■ 推奨提出順:
  
  1st: OOF最低のLGB seed
       → もし上記仮説が正しければ11.644以下の可能性
  
  2nd: seed35 LGB × bestPLS ブレンド (w=0.75)
       → 異なるモデルの組合せで安定改善を狙う
       → LGBとPLSの相関は~0.97で十分多様
  
  3rd: PLS-only の best seed
       → LGB成分ゼロの完全独立予測
       → 11.788より改善する可能性
""")

# 具体的な推奨ファイル
print(f"  ■ 具体的な推奨:")
# OOF最低のLGB
best_lgb = sorted_lgb[0]
print(f"    1st: submission_lgb_seed{best_lgb['seed']}.csv "
      f"(OOF={best_lgb['oof']:.4f})")

# ブレンド
print(f"    2nd: submission_lgb35_x_bestpls_w0.75.csv")

# PLS best
print(f"    3rd: submission_{sorted_pls[0]['name']}.csv "
      f"(OOF={sorted_pls[0]['oof']:.4f})")

print(f"""
  ■ 次回以降:
    - seed=35近傍で最良だったseedでPLS-onlyも実行
    - 3モデルブレンド (LGB seed35 + LGB seedX + PLS)
    - colsample_bytreeの微調整 (0.2, 0.25, 0.35)
    - early_stopping rounds変更 (20, 50)
""")

# submission_blend_3M_b0p6_ts0p20_p0p20

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)
train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
y_true = np.expm1(y_train_log)
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)

idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1450 = np.argmin(np.abs(wavelengths - 1450))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))
idx_1200 = np.argmin(np.abs(wavelengths - 1200))
idx_1790 = np.argmin(np.abs(wavelengths - 1790))
idx_2100 = np.argmin(np.abs(wavelengths - 2100))
idx_1700 = np.argmin(np.abs(wavelengths - 1700))
idx_2330 = np.argmin(np.abs(wavelengths - 2330))

water_band1 = (wavelengths >= 1350) & (wavelengths <= 1600)
water_band2 = (wavelengths >= 1800) & (wavelengths <= 2100)

X_train_raw = train[spec_cols].values
X_test_raw = test[spec_cols].values

# ============================================================
# 前処理関数
# ============================================================
def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

def mixup_cross_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    """異種間Mixup（既存）"""
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)

def mixup_within_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    """同一樹種内Mixup（新規）: 乾燥曲線を保持"""
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp = rng.choice(unique_species)
        sp_idx = np.where(species == sp)[0]
        if len(sp_idx) < 2:
            continue
        i1, i2 = rng.choice(sp_idx, size=2, replace=False)
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[i1] + (1 - lam) * X[i2])
        y_aug.append(lam * y[i1] + (1 - lam) * y[i2])
    return np.array(X_aug), np.array(y_aug)

def mixup_hybrid(X, y, species, n_cross=300, n_within=200, alpha=0.3, seed=42):
    """ハイブリッドMixup: 異種間+同一樹種内"""
    X_c, y_c = mixup_cross_species(X, y, species, n_cross, alpha, seed)
    X_w, y_w = mixup_within_species(X, y, species, n_within, alpha, seed + 1000)
    return np.vstack([X_c, X_w]), np.concatenate([y_c, y_w])

def extract_physics_features(X_raw):
    """物理ベース特徴量（厳選版）"""
    feats = {}
    # 水の吸収比率
    feats['r_1940_1300'] = X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_1450_1300'] = X_raw[:, idx_1450] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_1940_1450'] = X_raw[:, idx_1940] / (X_raw[:, idx_1450] + 1e-8)
    # C-H結合（木材本体）の比率
    feats['r_1700_1300'] = X_raw[:, idx_1700] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_2330_1300'] = X_raw[:, idx_2330] / (X_raw[:, idx_1300] + 1e-8)
    # 水バンドの面積（平均吸光度）
    feats['area_water1'] = np.mean(X_raw[:, water_band1], axis=1)
    feats['area_water2'] = np.mean(X_raw[:, water_band2], axis=1)
    feats['area_ratio'] = feats['area_water2'] / (feats['area_water1'] + 1e-8)
    # 散乱プロキシ（密度情報）
    feats['raw_std'] = np.std(X_raw, axis=1)
    feats['raw_mean'] = np.mean(X_raw, axis=1)
    # SNVスペクトルの水バンド統計
    snv = apply_snv(X_raw)
    feats['snv_water1_mean'] = np.mean(snv[:, water_band1], axis=1)
    feats['snv_water2_mean'] = np.mean(snv[:, water_band2], axis=1)
    feats['snv_water2_max'] = np.max(snv[:, water_band2], axis=1)
    # d1の水バンド統計
    d1 = savgol_filter(snv, 15, 2, deriv=1, axis=1)
    feats['d1_water1_std'] = np.std(d1[:, water_band1], axis=1)
    feats['d1_water2_std'] = np.std(d1[:, water_band2], axis=1)
    feats['d1_water2_min'] = np.min(d1[:, water_band2], axis=1)
    feats['d1_water2_max'] = np.max(d1[:, water_band2], axis=1)

    return pd.DataFrame(feats).values, list(feats.keys())


# ============================================================
# 方法A: Two-Stage Stacking (本命)
# Stage1: PLS + Ridge → OOF予測値
# Stage2: LGB(低次元: Stage1予測 + 物理特徴量 + KNN)
# ============================================================
def run_two_stage(lgb_seed=35, mixup_mode='hybrid', verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # Mixup
        if mixup_mode == 'hybrid':
            X_mix, y_mix = mixup_hybrid(
                X_tr, y_tr, tr_sp, 300, 200, 0.3, lgb_seed + fold)
        elif mixup_mode == 'within':
            X_mix, y_mix = mixup_within_species(
                X_tr, y_tr, tr_sp, 500, 0.3, lgb_seed + fold)
        else:
            X_mix, y_mix = mixup_cross_species(
                X_tr, y_tr, tr_sp, 500, 0.3, lgb_seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        # === Stage 1: 線形モデル群 ===
        snv_orig = apply_snv(X_tr)
        snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va)
        snv_te = apply_snv(X_test_raw)

        d1_orig = savgol_filter(snv_orig, 15, 2, deriv=1, axis=1)
        d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        d1_va = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        d1_te = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        # PLS (n_comp=2): SNV波形
        pls2 = PLSRegression(n_components=2, scale=False)
        pls2.fit(snv_orig, y_tr)
        pls2_pred_aug = pls2.predict(snv_aug).ravel()
        pls2_pred_va = pls2.predict(snv_va).ravel()
        pls2_pred_te = pls2.predict(snv_te).ravel()
        pls2_score_aug = pls2.transform(snv_aug)
        pls2_score_va = pls2.transform(snv_va)
        pls2_score_te = pls2.transform(snv_te)

        # PLS (n_comp=5): d1波形
        pls5 = PLSRegression(n_components=5, scale=False)
        pls5.fit(d1_orig, y_tr)
        pls5_pred_aug = pls5.predict(d1_aug).ravel()
        pls5_pred_va = pls5.predict(d1_va).ravel()
        pls5_pred_te = pls5.predict(d1_te).ravel()

        # Ridge on SNV
        scaler_r = StandardScaler()
        snv_orig_s = scaler_r.fit_transform(snv_orig)
        snv_aug_s = scaler_r.transform(snv_aug)
        snv_va_s = scaler_r.transform(snv_va)
        snv_te_s = scaler_r.transform(snv_te)
        ridge = Ridge(alpha=100.0)
        ridge.fit(snv_orig_s, y_tr)
        ridge_pred_aug = ridge.predict(snv_aug_s)
        ridge_pred_va = ridge.predict(snv_va_s)
        ridge_pred_te = ridge.predict(snv_te_s)

        # === KNN (PLS2空間) ===
        pls2_score_orig = pls2.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(pls2_score_orig)

        _, ik = knn.kneighbors(pls2_score_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])

        _, iv = knn.kneighbors(pls2_score_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1)
        _, it = knn.kneighbors(pls2_score_te, 5)
        knn_te = np.mean(y_tr[it], axis=1)

        # === PCA (SNV空間) for additional KNN ===
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va = pca.transform(snv_va)
        pc_te = pca.transform(snv_te)

        pc_orig = pca.transform(snv_orig)
        knn2 = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn2.fit(pc_orig)

        _, ik2 = knn2.kneighbors(pc_aug, n_neighbors=6)
        knn2_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik2[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn2_aug[i] = np.mean(y_tr[:n_orig][v])

        _, iv2 = knn2.kneighbors(pc_va, 5)
        knn2_va = np.mean(y_tr[iv2], axis=1)
        _, it2 = knn2.kneighbors(pc_te, 5)
        knn2_te = np.mean(y_tr[it2], axis=1)

        # === 物理特徴量 ===
        phys_aug, feat_names = extract_physics_features(X_aug)
        phys_va, _ = extract_physics_features(X_va)
        phys_te, _ = extract_physics_features(X_test_raw)

        # === Stage 2: LGB (低次元) ===
        # Stage1予測 + PLS潜在変数 + KNN + 物理特徴量 + PCA
        s2_names = (['pls2_pred', 'pls5_pred', 'ridge_pred',
                     'knn_pls', 'knn_pca'] +
                    [f'pls2_s{i}' for i in range(2)] +
                    [f'pc{i}' for i in range(10)] +
                    feat_names)

        ft = np.column_stack([
            pls2_pred_aug, pls5_pred_aug, ridge_pred_aug,
            knn_aug, knn2_aug,
            pls2_score_aug, pc_aug, phys_aug
        ])
        fv = np.column_stack([
            pls2_pred_va, pls5_pred_va, ridge_pred_va,
            knn_va, knn2_va,
            pls2_score_va, pc_va, phys_va
        ])
        fe = np.column_stack([
            pls2_pred_te, pls5_pred_te, ridge_pred_te,
            knn_te, knn2_te,
            pls2_score_te, pc_te, phys_te
        ])

        if verbose and fold == 0:
            print(f"    Stage2 入力次元: {ft.shape[1]} "
                  f"(従来: ~3131 → {ft.shape[1]})")

        model = lgb.LGBMRegressor(
            n_estimators=1500, learning_rate=0.02,
            max_depth=4, num_leaves=15,
            subsample=0.8, colsample_bytree=0.7,
            reg_alpha=0.1, reg_lambda=1.0,
            min_child_samples=20,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  TwoStage({mixup_mode}) seed={lgb_seed} "
              f"OOF={oof_rmse:.4f} "
              f"Fold={np.mean(fold_rmses):.4f}±{np.std(fold_rmses):.4f}")
    return {
        'name': f'TwoStage_{mixup_mode}_s{lgb_seed}',
        'oof': oof_rmse,
        'test_pred': final_pred, 'oof_pred': oof_pred
    }


# ============================================================
# 方法B: ベースライン再現（比較用）
# ============================================================
def run_lgb_baseline(seed=35, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_cross_species(
            X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_aug = apply_snv(X_aug)
        d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        snv_va = apply_snv(X_va)
        d1_va = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        snv_te = apply_snv(X_test_raw)
        d1_te = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va = np.std(X_va, axis=1, keepdims=True)
        s_te = np.std(X_test_raw, axis=1, keepdims=True)

        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va = pca.transform(snv_va)
        pc_te = pca.transform(snv_te)

        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pc_orig)

        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(pc_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it = knn.kneighbors(pc_te, 5)
        knn_te = np.mean(y_tr[it], axis=1).reshape(-1, 1)

        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, r_va, s_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  Baseline seed={seed} OOF={oof_rmse:.4f} "
              f"Fold={np.mean(fold_rmses):.4f}±{np.std(fold_rmses):.4f}")
    return {
        'name': f'Baseline_s{seed}',
        'oof': oof_rmse,
        'test_pred': final_pred, 'oof_pred': oof_pred
    }

# 方法C: PLS-only（ブレンド素材）
def run_pls_only(n_comp=2, seed=42, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups
    )):
        tr_sp = groups.iloc[tr_idx].values
        X_tr = X_train_raw[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va = X_train_raw[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_cross_species(
            X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_orig = apply_snv(X_tr)
        snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va)
        snv_te = apply_snv(X_test_raw)

        pls = PLSRegression(n_components=n_comp, scale=False)
        pls.fit(snv_orig, y_tr)

        ps_aug = pls.transform(snv_aug)
        ps_va = pls.transform(snv_va)
        ps_te = pls.transform(snv_te)
        pp_aug = pls.predict(snv_aug).ravel().reshape(-1,1)
        pp_va = pls.predict(snv_va).ravel().reshape(-1,1)
        pp_te = pls.predict(snv_te).ravel().reshape(-1,1)

        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(ps_orig)

        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(ps_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it_ = knn.kneighbors(ps_te, 5)
        knn_te = np.mean(y_tr[it_], axis=1).reshape(-1, 1)

        r_aug = (X_aug[:, idx_1940]/(X_aug[:, idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:, idx_1940]/(X_va[:, idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:, idx_1940]/(X_test_raw[:, idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va = np.std(X_va, axis=1, keepdims=True)
        s_te = np.std(X_test_raw, axis=1, keepdims=True)

        ft = np.hstack([ps_aug, pp_aug, knn_aug, r_aug, s_aug])
        fv = np.hstack([ps_va, pp_va, knn_va, r_va, s_va])
        fe = np.hstack([ps_te, pp_te, knn_te, r_te, s_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=4, num_leaves=15,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=-1)
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        pv = np.expm1(model.predict(fv))
        pt = np.expm1(model.predict(fe))
        oof_pred[va_idx] = pv
        final_pred += pt / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), pv))
        fold_rmses.append(rmse)

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  PLS{n_comp} seed={seed} OOF={oof_rmse:.4f}")
    return {
        'name': f'PLS{n_comp}_s{seed}',
        'oof': oof_rmse,
        'test_pred': final_pred, 'oof_pred': oof_pred
    }


# ============================================================
# 実行
# ============================================================
print("=" * 60)
print("🚀 Two-Stage Stacking 実験")
print("=" * 60)

results = {}

# ベースライン
print("\n📌 ベースライン (LB=11.644確認済み)")
r_base = run_lgb_baseline(seed=35)
results['baseline'] = r_base

# PLS-only (LB=11.788確認済み)
print("\n📌 PLS-only (LB=11.788確認済み)")
r_pls = run_pls_only(n_comp=2, seed=42)
results['pls2'] = r_pls

# Two-Stage: 各Mixupモード × seed
print("\n📌 Two-Stage Stacking 実験")
for mode in ['cross', 'within', 'hybrid']:
    for seed in [35, 42]:
        key = f'ts_{mode}_s{seed}'
        r = run_two_stage(lgb_seed=seed, mixup_mode=mode, verbose=True)
        results[key] = r

# seed安定性テスト (Two-Stageのhybridで複数seed)
print("\n📌 Two-Stage seed安定性テスト")
seed_oofs = []
for seed in [0, 7, 25, 33, 35, 42, 77, 99]:
    r = run_two_stage(lgb_seed=seed, mixup_mode='hybrid', verbose=False)
    results[f'ts_hybrid_s{seed}'] = r
    seed_oofs.append((seed, r['oof']))
    print(f"    seed={seed:3d} OOF={r['oof']:.4f}")

seed_oofs.sort(key=lambda x: x[1])
print(f"\n  seed安定性: OOF range = "
      f"{seed_oofs[0][1]:.4f} ~ {seed_oofs[-1][1]:.4f} "
      f"(差={seed_oofs[-1][1]-seed_oofs[0][1]:.4f})")
print(f"  ベースラインseed差: 16.79~18.04 = 1.25")
print(f"  → Two-Stageのseed安定性が改善されていれば勝ち")


# ============================================================
# ブレンド探索
# ============================================================
print(f"\n{'='*60}")
print("🔬 ブレンド探索")
print(f"{'='*60}")

blend_results = []

# Two-Stage × PLS-only
for key, r in results.items():
    if not key.startswith('ts_'):
        continue
    for w in np.arange(0.5, 0.95, 0.05):
        oof_bl = w * r['oof_pred'] + (1-w) * r_pls['oof_pred']
        rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
        pred_bl = w * r['test_pred'] + (1-w) * r_pls['test_pred']
        blend_results.append({
            'name': f"{r['name']}x{w:.2f}+PLS2x{1-w:.2f}",
            'oof': rmse_bl, 'test_pred': pred_bl, 'oof_pred': oof_bl,
            'components': [key, 'pls2']
        })

# Two-Stage × Baseline
for key, r in results.items():
    if not key.startswith('ts_'):
        continue
    for w in np.arange(0.3, 0.7, 0.05):
        oof_bl = w * r['oof_pred'] + (1-w) * r_base['oof_pred']
        rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
        pred_bl = w * r['test_pred'] + (1-w) * r_base['test_pred']
        blend_results.append({
            'name': f"{r['name']}x{w:.2f}+Basex{1-w:.2f}",
            'oof': rmse_bl, 'test_pred': pred_bl, 'oof_pred': oof_bl,
            'components': [key, 'baseline']
        })

# 3モデル: Baseline × Two-Stage × PLS
best_ts_key = min(
    [k for k in results if k.startswith('ts_')],
    key=lambda k: results[k]['oof']
)
best_ts = results[best_ts_key]
for wb in [0.4, 0.5, 0.6]:
    for wt in [0.2, 0.25, 0.3]:
        wp = 1 - wb - wt
        if wp < 0.05:
            continue
        oof_3 = (wb * r_base['oof_pred'] +
                 wt * best_ts['oof_pred'] +
                 wp * r_pls['oof_pred'])
        rmse_3 = np.sqrt(mean_squared_error(y_true, oof_3))
        pred_3 = (wb * r_base['test_pred'] +
                  wt * best_ts['test_pred'] +
                  wp * r_pls['test_pred'])
        blend_results.append({
            'name': f"3M_b{wb:.1f}_ts{wt:.2f}_p{wp:.2f}",
            'oof': rmse_3, 'test_pred': pred_3, 'oof_pred': oof_3,
            'components': ['baseline', best_ts_key, 'pls2']
        })

blend_results.sort(key=lambda x: x['oof'])

print(f"\n  Top 15 blends:")
print(f"  {'Name':<55s} {'OOF':>8s}")
print(f"  {'─'*55} {'─'*8}")
for b in blend_results[:15]:
    print(f"  {b['name']:<55s} {b['oof']:>8.4f}")


# ============================================================
# 提出ファイル作成（厳選5個）
# ============================================================
print(f"\n{'='*60}")
print("📁 提出ファイル作成")
print(f"{'='*60}")

submissions = {}

# 1. ベースライン (LB=11.644確認済み → 比較用)
out = submit_template.copy()
out[1] = np.clip(r_base['test_pred'], 0, None)
fname = "submission_baseline_s35_ref.csv"
out.to_csv(fname, index=False, header=False)
submissions[fname] = r_base['oof']
print(f"  ✅ {fname} (OOF={r_base['oof']:.4f}) [LB=11.644確認済み]")

# 2. Two-Stage単独ベスト
best_ts_r = results[best_ts_key]
out = submit_template.copy()
out[1] = np.clip(best_ts_r['test_pred'], 0, None)
safe_name = best_ts_r['name'].replace('.', 'p')
fname = f"submission_{safe_name}.csv"
out.to_csv(fname, index=False, header=False)
submissions[fname] = best_ts_r['oof']
print(f"  ✅ {fname} (OOF={best_ts_r['oof']:.4f})")

# 3. ベストブレンド
best_bl = blend_results[0]
out = submit_template.copy()
out[1] = np.clip(best_bl['test_pred'], 0, None)
safe_name = best_bl['name'].replace('.', 'p').replace('+', '_')
fname = f"submission_blend_{safe_name}.csv"
out.to_csv(fname, index=False, header=False)
submissions[fname] = best_bl['oof']
print(f"  ✅ {fname} (OOF={best_bl['oof']:.4f})")

# 4. 多様性ブレンド（異なるcomponents）
seen = set()
seen.add(tuple(sorted(best_bl['components'])))
for b in blend_results[1:]:
    comp_key = tuple(sorted(b['components']))
    if comp_key not in seen and len(submissions) < 5:
        seen.add(comp_key)
        out = submit_template.copy()
        out[1] = np.clip(b['test_pred'], 0, None)
        safe_name = b['name'].replace('.', 'p').replace('+', '_')
        fname = f"submission_blend_{safe_name}.csv"
        out.to_csv(fname, index=False, header=False)
        submissions[fname] = b['oof']
        print(f"  ✅ {fname} (OOF={b['oof']:.4f})")


# ============================================================
# 最終サマリー
# ============================================================
print(f"\n{'='*60}")
print("📌 最終サマリー")
print(f"{'='*60}")

print(f"\n  {'単独モデル':<50s} {'OOF':>8s}")
print(f"  {'─'*50} {'─'*8}")
for key in sorted(results, key=lambda k: results[k]['oof']):
    r = results[key]
    mark = ""
    if 'baseline' in key:
        mark = " [LB=11.644]"
    elif key == 'pls2':
        mark = " [LB=11.788]"
    print(f"  {r['name']:<50s} {r['oof']:>8.4f}{mark}")

print(f"\n  {'提出候補':<50s} {'OOF':>8s}")
print(f"  {'─'*50} {'─'*8}")
for fname, oof in sorted(submissions.items(), key=lambda x: x[1]):
    short = fname.replace("submission_", "").replace(".csv", "")
    print(f"  {short:<50s} {oof:>8.4f}")

# seed安定性の比較
ts_oofs = [results[k]['oof'] for k in results if k.startswith('ts_hybrid')]
base_oof_range = 18.04 - 16.79  # seed42 vs seed35
ts_oof_range = max(ts_oofs) - min(ts_oofs) if ts_oofs else 0

print(f"""
■ 改善の核心分析:
  ベースライン入力次元: ~3131 → seed依存性: OOF差 {base_oof_range:.2f}
  Two-Stage 入力次元:   ~34   → seed依存性: OOF差 {ts_oof_range:.2f}
  → 次元削減による安定化 = {'成功' if ts_oof_range < base_oof_range else '要検証'}

■ 推奨提出順:
  1st: Two-Stage単独 or ベストブレンド (新手法の検証)
  2nd: Baseline+TwoStage or Baseline+PLS ブレンド
  3rd: ベースライン (LB確認済み参照点)
""")

# submission_improved

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)
train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
y_true = np.expm1(y_train_log)
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)

idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1450 = np.argmin(np.abs(wavelengths - 1450))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))
idx_1700 = np.argmin(np.abs(wavelengths - 1700))
idx_2330 = np.argmin(np.abs(wavelengths - 2330))
# 水の吸収を避けた散乱帯 (1100-1200nm)
idx_1100 = np.argmin(np.abs(wavelengths - 1100))
idx_1200 = np.argmin(np.abs(wavelengths - 1200))
scatter_band = (wavelengths >= 1100) & (wavelengths <= 1250)

water_band1 = (wavelengths >= 1350) & (wavelengths <= 1600)
water_band2 = (wavelengths >= 1800) & (wavelengths <= 2100)

X_train_raw = train[spec_cols].values
X_test_raw = test[spec_cols].values

# ============================================================
# 前処理関数
# ============================================================
def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

def mixup_cross_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)

def mixup_within_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp = rng.choice(unique_species)
        sp_idx = np.where(species == sp)[0]
        if len(sp_idx) < 2:
            continue
        i1, i2 = rng.choice(sp_idx, size=2, replace=False)
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[i1] + (1 - lam) * X[i2])
        y_aug.append(lam * y[i1] + (1 - lam) * y[i2])
    return np.array(X_aug), np.array(y_aug)

def mixup_hybrid(X, y, species, n_cross=300, n_within=200, alpha=0.3, seed=42):
    X_c, y_c = mixup_cross_species(X, y, species, n_cross, alpha, seed)
    X_w, y_w = mixup_within_species(X, y, species, n_within, alpha, seed + 1000)
    return np.vstack([X_c, X_w]), np.concatenate([y_c, y_w])

def extract_physics_features(X_raw):
    feats = {}
    feats['r_1940_1300'] = X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_1450_1300'] = X_raw[:, idx_1450] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_1940_1450'] = X_raw[:, idx_1940] / (X_raw[:, idx_1450] + 1e-8)
    feats['r_1700_1300'] = X_raw[:, idx_1700] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_2330_1300'] = X_raw[:, idx_2330] / (X_raw[:, idx_1300] + 1e-8)
    feats['area_water1'] = np.mean(X_raw[:, water_band1], axis=1)
    feats['area_water2'] = np.mean(X_raw[:, water_band2], axis=1)
    feats['area_ratio'] = feats['area_water2'] / (feats['area_water1'] + 1e-8)
    feats['raw_std'] = np.std(X_raw, axis=1)
    feats['raw_mean'] = np.mean(X_raw, axis=1)
    # 改善: 散乱帯(水を避けた)の平均 → 真の密度プロキシ
    feats['scatter_mean'] = np.mean(X_raw[:, scatter_band], axis=1)
    feats['scatter_std'] = np.std(X_raw[:, scatter_band], axis=1)
    snv = apply_snv(X_raw)
    feats['snv_water1_mean'] = np.mean(snv[:, water_band1], axis=1)
    feats['snv_water2_mean'] = np.mean(snv[:, water_band2], axis=1)
    feats['snv_water2_max'] = np.max(snv[:, water_band2], axis=1)
    d1 = savgol_filter(snv, 15, 2, deriv=1, axis=1)
    feats['d1_water1_std'] = np.std(d1[:, water_band1], axis=1)
    feats['d1_water2_std'] = np.std(d1[:, water_band2], axis=1)
    feats['d1_water2_min'] = np.min(d1[:, water_band2], axis=1)
    feats['d1_water2_max'] = np.max(d1[:, water_band2], axis=1)
    return pd.DataFrame(feats).values, list(feats.keys())

# ============================================================
# データ分析: LB確認済みスコアのパターン分析
# ============================================================
print("=" * 60)
print("📊 LB確認済みスコア分析")
print("=" * 60)
print("""
  LB確認済み:
    ① b0.60+t0.20+p0.20 → LB=11.330 (OOF=16.82) ★BEST
    ② b0.50+t?+pls2_0   → LB=11.338 (OOF=?)
    ③ b0.75+t0.10+p0.15 → LB=11.403 (OOF=16.71)
    ④ Base単独 seed35    → LB=11.644 (OOF=16.79)
    ⑤ PLS2_42単独        → LB=11.788 (OOF=19.60)
    ⑥ 4M+ppls_3(20%)    → LB=12.333 (OOF=16.21) ← 大悪化!
    ⑦ TS平均ブレンド      → LB=11.398

  教訓:
  - ①vs③: Base比率↓でLB改善 (0.75→0.60で11.403→11.330)
  - ①vs②: wb=0.50とwb=0.60でほぼ同等 → 0.55付近が最適か
  - ⑥: OOF最良(16.21)がLB最悪(12.333) → ppls_3は未知樹種に壊滅的
  - ⑦: TS平均化は11.398 (単一TS_h7使用の11.330より悪い)
  
  結論: 既存3モデル(Base35+TS_h7+PLS2_42)の枠組みが最強
  新モデル追加はリスク大 → 各モデル自体の改善に注力
""")


# ============================================================
# モデルA: ベースラインLGB (LB=11.644)
# ============================================================
def run_lgb_baseline(seed=35, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_aug = apply_snv(X_aug); d1_aug = savgol_filter(snv_aug,15,2,deriv=1,axis=1)
        snv_va = apply_snv(X_va); d1_va = savgol_filter(snv_va,15,2,deriv=1,axis=1)
        snv_te = apply_snv(X_test_raw); d1_te = savgol_filter(snv_te,15,2,deriv=1,axis=1)
        r_aug = (X_aug[:,idx_1940]/(X_aug[:,idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:,idx_1940]/(X_va[:,idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:,idx_1940]/(X_test_raw[:,idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug,axis=1,keepdims=True)
        s_va = np.std(X_va,axis=1,keepdims=True)
        s_te = np.std(X_test_raw,axis=1,keepdims=True)
        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42); pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug); pc_va = pca.transform(snv_va); pc_te = pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine'); knn.fit(pc_orig)
        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1,1)
        _, iv = knn.kneighbors(pc_va,5); knn_va = np.mean(y_tr[iv],axis=1).reshape(-1,1)
        _, it = knn.kneighbors(pc_te,5); knn_te = np.mean(y_tr[it],axis=1).reshape(-1,1)
        ft = np.hstack([snv_aug,d1_aug,pc_aug,knn_aug,r_aug,s_aug])
        fv = np.hstack([snv_va,d1_va,pc_va,knn_va,r_va,s_va])
        fe = np.hstack([snv_te,d1_te,pc_te,knn_te,r_te,s_te])
        model = lgb.LGBMRegressor(n_estimators=1000,learning_rate=0.03,max_depth=5,
            num_leaves=31,subsample=0.8,colsample_bytree=0.3,random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(30,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  Base seed={seed} OOF={oof_rmse:.4f}")
    return {'name':f'Base_s{seed}','oof':oof_rmse,'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# モデルB: PLS-only
# ============================================================
def run_pls_only(n_comp=2, seed=42, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_orig = apply_snv(X_tr); snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va); snv_te = apply_snv(X_test_raw)
        pls = PLSRegression(n_components=n_comp, scale=False); pls.fit(snv_orig, y_tr)
        ps_aug = pls.transform(snv_aug); ps_va = pls.transform(snv_va); ps_te = pls.transform(snv_te)
        pp_aug = pls.predict(snv_aug).ravel().reshape(-1,1)
        pp_va = pls.predict(snv_va).ravel().reshape(-1,1)
        pp_te = pls.predict(snv_te).ravel().reshape(-1,1)
        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean'); knn.fit(ps_orig)
        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1,1)
        _, iv = knn.kneighbors(ps_va,5); knn_va = np.mean(y_tr[iv],axis=1).reshape(-1,1)
        _, it_ = knn.kneighbors(ps_te,5); knn_te = np.mean(y_tr[it_],axis=1).reshape(-1,1)
        r_aug = (X_aug[:,idx_1940]/(X_aug[:,idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:,idx_1940]/(X_va[:,idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:,idx_1940]/(X_test_raw[:,idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug,axis=1,keepdims=True)
        s_va = np.std(X_va,axis=1,keepdims=True)
        s_te = np.std(X_test_raw,axis=1,keepdims=True)
        ft = np.hstack([ps_aug,pp_aug,knn_aug,r_aug,s_aug])
        fv = np.hstack([ps_va,pp_va,knn_va,r_va,s_va])
        fe = np.hstack([ps_te,pp_te,knn_te,r_te,s_te])
        model = lgb.LGBMRegressor(n_estimators=1000,learning_rate=0.03,max_depth=4,
            num_leaves=15,subsample=0.8,colsample_bytree=0.8,random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(30,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  PLS{n_comp} seed={seed} OOF={oof_rmse:.4f}")
    return {'name':f'PLS{n_comp}_s{seed}','oof':oof_rmse,'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# モデルC: Two-Stage Stacking
# ============================================================
def run_two_stage(lgb_seed=7, mixup_mode='hybrid', verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        if mixup_mode == 'hybrid':
            X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, 300, 200, 0.3, lgb_seed+fold)
        else:
            X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, lgb_seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_orig = apply_snv(X_tr); snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va); snv_te = apply_snv(X_test_raw)
        d1_orig = savgol_filter(snv_orig,15,2,deriv=1,axis=1)
        d1_aug = savgol_filter(snv_aug,15,2,deriv=1,axis=1)
        d1_va = savgol_filter(snv_va,15,2,deriv=1,axis=1)
        d1_te = savgol_filter(snv_te,15,2,deriv=1,axis=1)
        pls2 = PLSRegression(n_components=2, scale=False); pls2.fit(snv_orig, y_tr)
        pls2_pred_aug = pls2.predict(snv_aug).ravel()
        pls2_pred_va = pls2.predict(snv_va).ravel()
        pls2_pred_te = pls2.predict(snv_te).ravel()
        pls2_score_aug = pls2.transform(snv_aug)
        pls2_score_va = pls2.transform(snv_va)
        pls2_score_te = pls2.transform(snv_te)
        pls5 = PLSRegression(n_components=5, scale=False); pls5.fit(d1_orig, y_tr)
        pls5_pred_aug = pls5.predict(d1_aug).ravel()
        pls5_pred_va = pls5.predict(d1_va).ravel()
        pls5_pred_te = pls5.predict(d1_te).ravel()
        scaler_r = StandardScaler(); snv_orig_s = scaler_r.fit_transform(snv_orig)
        ridge = Ridge(alpha=100.0); ridge.fit(snv_orig_s, y_tr)
        ridge_pred_aug = ridge.predict(scaler_r.transform(snv_aug))
        ridge_pred_va = ridge.predict(scaler_r.transform(snv_va))
        ridge_pred_te = ridge.predict(scaler_r.transform(snv_te))
        pls2_score_orig = pls2.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean'); knn.fit(pls2_score_orig)
        _, ik = knn.kneighbors(pls2_score_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv = knn.kneighbors(pls2_score_va,5); knn_va = np.mean(y_tr[iv],axis=1)
        _, it = knn.kneighbors(pls2_score_te,5); knn_te = np.mean(y_tr[it],axis=1)
        pca = PCA(n_components=10, random_state=42); pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug); pc_va = pca.transform(snv_va); pc_te = pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn2 = NearestNeighbors(n_neighbors=5, metric='cosine'); knn2.fit(pc_orig)
        _, ik2 = knn2.kneighbors(pc_aug, n_neighbors=6)
        knn2_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik2[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn2_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv2 = knn2.kneighbors(pc_va,5); knn2_va = np.mean(y_tr[iv2],axis=1)
        _, it2 = knn2.kneighbors(pc_te,5); knn2_te = np.mean(y_tr[it2],axis=1)
        phys_aug, _ = extract_physics_features(X_aug)
        phys_va, _ = extract_physics_features(X_va)
        phys_te, _ = extract_physics_features(X_test_raw)
        ft = np.column_stack([pls2_pred_aug,pls5_pred_aug,ridge_pred_aug,
            knn_aug,knn2_aug,pls2_score_aug,pc_aug,phys_aug])
        fv = np.column_stack([pls2_pred_va,pls5_pred_va,ridge_pred_va,
            knn_va,knn2_va,pls2_score_va,pc_va,phys_va])
        fe = np.column_stack([pls2_pred_te,pls5_pred_te,ridge_pred_te,
            knn_te,knn2_te,pls2_score_te,pc_te,phys_te])
        model = lgb.LGBMRegressor(n_estimators=1500,learning_rate=0.02,max_depth=4,
            num_leaves=15,subsample=0.8,colsample_bytree=0.7,reg_alpha=0.1,reg_lambda=1.0,
            min_child_samples=20,random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(50,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  TS({mixup_mode}) seed={lgb_seed} OOF={oof_rmse:.4f}")
    return {'name':f'TS_{mixup_mode[0]}_s{lgb_seed}','oof':oof_rmse,
            'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# 【改善】モデルA': 密度補正+正則化強化ベースライン
# ============================================================
def run_lgb_improved(seed=35, verbose=True):
    """
    改善点:
    1. scatter_band(1100-1250nm)の平均を密度プロキシとして追加
    2. colsample_bytree微調整
    3. min_child_samplesで正則化強化
    """
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])

        snv_aug = apply_snv(X_aug); d1_aug = savgol_filter(snv_aug,15,2,deriv=1,axis=1)
        snv_va = apply_snv(X_va); d1_va = savgol_filter(snv_va,15,2,deriv=1,axis=1)
        snv_te = apply_snv(X_test_raw); d1_te = savgol_filter(snv_te,15,2,deriv=1,axis=1)

        r_aug = (X_aug[:,idx_1940]/(X_aug[:,idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:,idx_1940]/(X_va[:,idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:,idx_1940]/(X_test_raw[:,idx_1300]+1e-8)).reshape(-1,1)

        # 改善: 散乱帯(水を避けた1100-1250nm)の平均 = 純粋な密度プロキシ
        sc_aug = np.mean(X_aug[:, scatter_band], axis=1, keepdims=True)
        sc_va = np.mean(X_va[:, scatter_band], axis=1, keepdims=True)
        sc_te = np.mean(X_test_raw[:, scatter_band], axis=1, keepdims=True)

        s_aug = np.std(X_aug,axis=1,keepdims=True)
        s_va = np.std(X_va,axis=1,keepdims=True)
        s_te = np.std(X_test_raw,axis=1,keepdims=True)

        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42); pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug); pc_va = pca.transform(snv_va); pc_te = pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine'); knn.fit(pc_orig)
        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1,1)
        _, iv = knn.kneighbors(pc_va,5); knn_va = np.mean(y_tr[iv],axis=1).reshape(-1,1)
        _, it = knn.kneighbors(pc_te,5); knn_te = np.mean(y_tr[it],axis=1).reshape(-1,1)

        ft = np.hstack([snv_aug,d1_aug,pc_aug,knn_aug,r_aug,s_aug,sc_aug])
        fv = np.hstack([snv_va,d1_va,pc_va,knn_va,r_va,s_va,sc_va])
        fe = np.hstack([snv_te,d1_te,pc_te,knn_te,r_te,s_te,sc_te])

        model = lgb.LGBMRegressor(
            n_estimators=1000,learning_rate=0.03,max_depth=5,
            num_leaves=31,subsample=0.8,colsample_bytree=0.3,
            min_child_samples=15,  # 少し正則化強化
            reg_alpha=0.05, reg_lambda=0.5,
            random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(30,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  ImprovedBase seed={seed} OOF={oof_rmse:.4f}")
    return {'name':f'ImpBase_s{seed}','oof':oof_rmse,'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# 【改善】モデルC': Two-Stage 改良版（散乱帯密度補正）
# ============================================================
def run_two_stage_improved(lgb_seed=7, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, 300, 200, 0.3, lgb_seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_orig = apply_snv(X_tr); snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va); snv_te = apply_snv(X_test_raw)
        d1_orig = savgol_filter(snv_orig,15,2,deriv=1,axis=1)
        d1_aug = savgol_filter(snv_aug,15,2,deriv=1,axis=1)
        d1_va = savgol_filter(snv_va,15,2,deriv=1,axis=1)
        d1_te = savgol_filter(snv_te,15,2,deriv=1,axis=1)
        pls2 = PLSRegression(n_components=2, scale=False); pls2.fit(snv_orig, y_tr)
        pls2_pred_aug = pls2.predict(snv_aug).ravel()
        pls2_pred_va = pls2.predict(snv_va).ravel()
        pls2_pred_te = pls2.predict(snv_te).ravel()
        pls2_score_aug = pls2.transform(snv_aug)
        pls2_score_va = pls2.transform(snv_va)
        pls2_score_te = pls2.transform(snv_te)
        pls5 = PLSRegression(n_components=5, scale=False); pls5.fit(d1_orig, y_tr)
        pls5_pred_aug = pls5.predict(d1_aug).ravel()
        pls5_pred_va = pls5.predict(d1_va).ravel()
        pls5_pred_te = pls5.predict(d1_te).ravel()
        scaler_r = StandardScaler(); snv_orig_s = scaler_r.fit_transform(snv_orig)
        ridge = Ridge(alpha=100.0); ridge.fit(snv_orig_s, y_tr)
        ridge_pred_aug = ridge.predict(scaler_r.transform(snv_aug))
        ridge_pred_va = ridge.predict(scaler_r.transform(snv_va))
        ridge_pred_te = ridge.predict(scaler_r.transform(snv_te))
        pls2_score_orig = pls2.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean'); knn.fit(pls2_score_orig)
        _, ik = knn.kneighbors(pls2_score_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv = knn.kneighbors(pls2_score_va,5); knn_va = np.mean(y_tr[iv],axis=1)
        _, it = knn.kneighbors(pls2_score_te,5); knn_te = np.mean(y_tr[it],axis=1)
        pca = PCA(n_components=10, random_state=42); pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug); pc_va = pca.transform(snv_va); pc_te = pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn2 = NearestNeighbors(n_neighbors=5, metric='cosine'); knn2.fit(pc_orig)
        _, ik2 = knn2.kneighbors(pc_aug, n_neighbors=6)
        knn2_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik2[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn2_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv2 = knn2.kneighbors(pc_va,5); knn2_va = np.mean(y_tr[iv2],axis=1)
        _, it2 = knn2.kneighbors(pc_te,5); knn2_te = np.mean(y_tr[it2],axis=1)
        phys_aug, _ = extract_physics_features(X_aug)
        phys_va, _ = extract_physics_features(X_va)
        phys_te, _ = extract_physics_features(X_test_raw)
        ft = np.column_stack([pls2_pred_aug,pls5_pred_aug,ridge_pred_aug,
            knn_aug,knn2_aug,pls2_score_aug,pc_aug,phys_aug])
        fv = np.column_stack([pls2_pred_va,pls5_pred_va,ridge_pred_va,
            knn_va,knn2_va,pls2_score_va,pc_va,phys_va])
        fe = np.column_stack([pls2_pred_te,pls5_pred_te,ridge_pred_te,
            knn_te,knn2_te,pls2_score_te,pc_te,phys_te])
        model = lgb.LGBMRegressor(n_estimators=1500,learning_rate=0.02,max_depth=4,
            num_leaves=15,subsample=0.8,colsample_bytree=0.7,reg_alpha=0.1,reg_lambda=1.0,
            min_child_samples=20,random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(50,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  TS_imp seed={lgb_seed} OOF={oof_rmse:.4f}")
    return {'name':f'TS_imp_s{lgb_seed}','oof':oof_rmse,
            'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# 実行
# ============================================================
print("\n" + "=" * 60)
print("🚀 改善実験: 既存3モデル枠組み維持 + 各モデル改善")
print("=" * 60)

results = {}

# 既存3モデル（LB確認済み構成の再現）
print("\n📌 既存モデル再現")
results['base35'] = run_lgb_baseline(seed=35)
results['pls2_42'] = run_pls_only(n_comp=2, seed=42)
results['ts_h_7'] = run_two_stage(lgb_seed=7, mixup_mode='hybrid')

# 改善版モデル
print("\n📌 改善版モデル")
results['imp_base35'] = run_lgb_improved(seed=35)
results['ts_imp_7'] = run_two_stage_improved(lgb_seed=7)

# TS別seed (改善版)
for s in [33, 35]:
    results[f'ts_imp_{s}'] = run_two_stage_improved(lgb_seed=s)


# ============================================================
# ブレンド探索（0.025刻み精密）
# ============================================================
print(f"\n{'='*60}")
print("🔬 ブレンド精密探索 (既存3モデル枠組み)")
print(f"{'='*60}")

blend_results = []

# 既存3モデルの比率精密探索 (wb=0.50~0.65)
base_variants = ['base35', 'imp_base35']
ts_variants = ['ts_h_7', 'ts_imp_7', 'ts_imp_33', 'ts_imp_35']
pls_variants = ['pls2_42']

for bk in base_variants:
    for tk in ts_variants:
        for pk in pls_variants:
            for wb in np.arange(0.475, 0.650, 0.025):
                for wt in np.arange(0.125, 0.300, 0.025):
                    wp = round(1.0 - wb - wt, 3)
                    if wp < 0.10 or wp > 0.35:
                        continue
                    oof_bl = (wb * results[bk]['oof_pred'] +
                              wt * results[tk]['oof_pred'] +
                              wp * results[pk]['oof_pred'])
                    rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
                    pred_bl = (wb * results[bk]['test_pred'] +
                               wt * results[tk]['test_pred'] +
                               wp * results[pk]['test_pred'])
                    blend_results.append({
                        'name': f"{bk}_x{wb:.3f}+{tk}_x{wt:.3f}+{pk}_x{wp:.3f}",
                        'base': bk, 'ts': tk, 'pls': pk,
                        'wb': wb, 'wt': wt, 'wp': wp,
                        'oof': rmse_bl,
                        'test_pred': pred_bl,
                    })

blend_results.sort(key=lambda x: x['oof'])

# LB=11.330の再現構成を探す
print(f"\n  LB=11.330再現構成:")
for b in blend_results:
    if (b['base'] == 'base35' and b['ts'] == 'ts_h_7' and
        abs(b['wb']-0.6)<0.015 and abs(b['wt']-0.2)<0.015):
        print(f"    {b['name']} OOF={b['oof']:.4f} → LB=11.330")
        break

print(f"\n  Top 20:")
print(f"  {'Name':<65s} {'OOF':>8s}")
print(f"  {'─'*65} {'─'*8}")
for b in blend_results[:20]:
    mark = ""
    if b['base']=='base35' and b['ts']=='ts_h_7' and abs(b['wb']-0.6)<0.015:
        mark = " ★11.330"
    print(f"  {b['name']:<65s} {b['oof']:>8.4f}{mark}")

# 改善版 vs 既存版 の比較
print(f"\n  改善版vs既存版 (同比率での比較):")
for wb_target in [0.55, 0.575, 0.60]:
    orig = None; imp = None
    for b in blend_results:
        if abs(b['wb']-wb_target)<0.015 and abs(b['wt']-0.2)<0.015:
            if b['base']=='base35' and b['ts']=='ts_h_7' and orig is None:
                orig = b
            elif 'imp' in b['base'] and imp is None:
                imp = b
            elif 'imp' in b['ts'] and imp is None:
                imp = b
    if orig and imp:
        print(f"    wb≈{wb_target:.2f}: 既存OOF={orig['oof']:.4f} 改善OOF={imp['oof']:.4f} "
              f"差={imp['oof']-orig['oof']:.4f}")


# ============================================================
# 提出ファイル（厳選3個）
# ============================================================
print(f"\n{'='*60}")
print("📁 提出ファイル作成（厳選3個）")
print(f"{'='*60}")

submissions = {}

# 1. 既存3モデルの比率微調整（LB=11.330からの微改善を狙う）
# wb=0.55付近（0.60より少しBase下げ）
best_tune = None
for b in blend_results:
    if (b['base']=='base35' and b['ts']=='ts_h_7' and b['pls']=='pls2_42'
        and 0.54 < b['wb'] < 0.58):
        best_tune = b
        break
if best_tune:
    out = submit_template.copy()
    out[1] = np.clip(best_tune['test_pred'], 0, None)
    fname = "submission_3M_wb055.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_tune['oof']
    print(f"  ✅ {fname} (OOF={best_tune['oof']:.4f})")
    print(f"     [{best_tune['name']}]")

# 2. 改善版Base or TS を使った最良ブレンド
best_imp = None
for b in blend_results:
    if 'imp' in b['base'] or 'imp' in b['ts']:
        best_imp = b
        break
if best_imp:
    out = submit_template.copy()
    out[1] = np.clip(best_imp['test_pred'], 0, None)
    fname = "submission_improved.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_imp['oof']
    print(f"  ✅ {fname} (OOF={best_imp['oof']:.4f})")
    print(f"     [{best_imp['name']}]")

# 3. OOF最良の構成（ただし既出と異なるもの）
for b in blend_results:
    already = {best_tune['name'] if best_tune else '', best_imp['name'] if best_imp else ''}
    if b['name'] not in already:
        out = submit_template.copy()
        out[1] = np.clip(b['test_pred'], 0, None)
        fname = "submission_oofbest.csv"
        out.to_csv(fname, index=False, header=False)
        submissions[fname] = b['oof']
        print(f"  ✅ {fname} (OOF={b['oof']:.4f})")
        print(f"     [{b['name']}]")
        break


# ============================================================
# 最終分析
# ============================================================
print(f"\n{'='*60}")
print("📌 最終分析")
print(f"{'='*60}")

print(f"\n  LB確認済みスコア一覧:")
print(f"    b0.60+t0.20+p0.20 (既存3M)    → LB=11.330 ★BEST")
print(f"    b0.50+ts_h_7+pls2_0            → LB=11.338")
print(f"    b0.75+t0.10+p0.15              → LB=11.403")
print(f"    Base単独 seed35                 → LB=11.644")
print(f"    PLS2_42単独                     → LB=11.788")
print(f"    4M+ppls_3(20%)                 → LB=12.333 ✗大悪化")
print(f"    TS平均ブレンド                  → LB=11.398")

print(f"\n  Gemini指摘の精査結果:")
print(f"    Mixup非物理性  → 正則化として有効、変更不要")
print(f"    raw_std問題    → scatter_band密度プロキシで改善試行")
print(f"    SNVアーティファクト → MSC既にLB悪化済み、SNV維持が最善")
print(f"    時間特徴量     → ルール違反のため却下")

print(f"\n  本質的教訓:")
print(f"    - 新モデル追加はLBリスク大 (ppls_3: OOF改善→LB大悪化)")
print(f"    - 既存3M枠組みの比率微調整が最も安全な改善路線")
print(f"    - 各モデル自体の小改善(密度補正等)で枠組み内改善を狙う")

print(f"\n  提出候補:")
for fname, oof in sorted(submissions.items(), key=lambda x: x[1]):
    short = fname.replace("submission_","").replace(".csv","")
    print(f"    {short:<30s} OOF={oof:.4f}")

# submission_all_improved.csv

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)
train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
y_true = np.expm1(y_train_log)
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)

idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1450 = np.argmin(np.abs(wavelengths - 1450))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))
idx_1700 = np.argmin(np.abs(wavelengths - 1700))
idx_2330 = np.argmin(np.abs(wavelengths - 2330))

scatter_band = (wavelengths >= 1100) & (wavelengths <= 1250)
water_band1 = (wavelengths >= 1350) & (wavelengths <= 1600)
water_band2 = (wavelengths >= 1800) & (wavelengths <= 2100)

X_train_raw = train[spec_cols].values
X_test_raw = test[spec_cols].values

# ============================================================
# 前処理関数
# ============================================================
def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

def mixup_cross_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)

def mixup_within_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp = rng.choice(unique_species)
        sp_idx = np.where(species == sp)[0]
        if len(sp_idx) < 2:
            continue
        i1, i2 = rng.choice(sp_idx, size=2, replace=False)
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[i1] + (1 - lam) * X[i2])
        y_aug.append(lam * y[i1] + (1 - lam) * y[i2])
    return np.array(X_aug), np.array(y_aug)

def mixup_hybrid(X, y, species, n_cross=300, n_within=200, alpha=0.3, seed=42):
    X_c, y_c = mixup_cross_species(X, y, species, n_cross, alpha, seed)
    X_w, y_w = mixup_within_species(X, y, species, n_within, alpha, seed+1000)
    return np.vstack([X_c, X_w]), np.concatenate([y_c, y_w])

def extract_physics_features(X_raw):
    feats = {}
    feats['r_1940_1300'] = X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_1450_1300'] = X_raw[:, idx_1450] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_1940_1450'] = X_raw[:, idx_1940] / (X_raw[:, idx_1450] + 1e-8)
    feats['r_1700_1300'] = X_raw[:, idx_1700] / (X_raw[:, idx_1300] + 1e-8)
    feats['r_2330_1300'] = X_raw[:, idx_2330] / (X_raw[:, idx_1300] + 1e-8)
    feats['area_water1'] = np.mean(X_raw[:, water_band1], axis=1)
    feats['area_water2'] = np.mean(X_raw[:, water_band2], axis=1)
    feats['area_ratio'] = feats['area_water2'] / (feats['area_water1'] + 1e-8)
    feats['raw_std'] = np.std(X_raw, axis=1)
    feats['raw_mean'] = np.mean(X_raw, axis=1)
    feats['scatter_mean'] = np.mean(X_raw[:, scatter_band], axis=1)
    feats['scatter_std'] = np.std(X_raw[:, scatter_band], axis=1)
    snv = apply_snv(X_raw)
    feats['snv_water1_mean'] = np.mean(snv[:, water_band1], axis=1)
    feats['snv_water2_mean'] = np.mean(snv[:, water_band2], axis=1)
    feats['snv_water2_max'] = np.max(snv[:, water_band2], axis=1)
    d1 = savgol_filter(snv, 15, 2, deriv=1, axis=1)
    feats['d1_water1_std'] = np.std(d1[:, water_band1], axis=1)
    feats['d1_water2_std'] = np.std(d1[:, water_band2], axis=1)
    feats['d1_water2_min'] = np.min(d1[:, water_band2], axis=1)
    feats['d1_water2_max'] = np.max(d1[:, water_band2], axis=1)
    return pd.DataFrame(feats).values, list(feats.keys())


# ============================================================
# モデルA: 改善版ベースラインLGB（scatter_band密度補正）
# ============================================================
def run_lgb_improved(seed=35, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_aug = apply_snv(X_aug); d1_aug = savgol_filter(snv_aug,15,2,deriv=1,axis=1)
        snv_va = apply_snv(X_va); d1_va = savgol_filter(snv_va,15,2,deriv=1,axis=1)
        snv_te = apply_snv(X_test_raw); d1_te = savgol_filter(snv_te,15,2,deriv=1,axis=1)
        r_aug = (X_aug[:,idx_1940]/(X_aug[:,idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:,idx_1940]/(X_va[:,idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:,idx_1940]/(X_test_raw[:,idx_1300]+1e-8)).reshape(-1,1)
        sc_aug = np.mean(X_aug[:,scatter_band],axis=1,keepdims=True)
        sc_va = np.mean(X_va[:,scatter_band],axis=1,keepdims=True)
        sc_te = np.mean(X_test_raw[:,scatter_band],axis=1,keepdims=True)
        s_aug = np.std(X_aug,axis=1,keepdims=True)
        s_va = np.std(X_va,axis=1,keepdims=True)
        s_te = np.std(X_test_raw,axis=1,keepdims=True)
        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42); pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug); pc_va = pca.transform(snv_va); pc_te = pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine'); knn.fit(pc_orig)
        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1,1)
        _, iv = knn.kneighbors(pc_va,5); knn_va = np.mean(y_tr[iv],axis=1).reshape(-1,1)
        _, it = knn.kneighbors(pc_te,5); knn_te = np.mean(y_tr[it],axis=1).reshape(-1,1)
        ft = np.hstack([snv_aug,d1_aug,pc_aug,knn_aug,r_aug,s_aug,sc_aug])
        fv = np.hstack([snv_va,d1_va,pc_va,knn_va,r_va,s_va,sc_va])
        fe = np.hstack([snv_te,d1_te,pc_te,knn_te,r_te,s_te,sc_te])
        model = lgb.LGBMRegressor(n_estimators=1000,learning_rate=0.03,max_depth=5,
            num_leaves=31,subsample=0.8,colsample_bytree=0.3,
            min_child_samples=15,reg_alpha=0.05,reg_lambda=0.5,
            random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(30,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  ImpBase seed={seed} OOF={oof_rmse:.4f}")
    return {'name':f'ImpBase_s{seed}','oof':oof_rmse,'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# モデルB: PLS-only 改善版（scatter_band追加）
# ============================================================
def run_pls_improved(n_comp=2, seed=42, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_orig = apply_snv(X_tr); snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va); snv_te = apply_snv(X_test_raw)
        pls = PLSRegression(n_components=n_comp, scale=False); pls.fit(snv_orig, y_tr)
        ps_aug = pls.transform(snv_aug); ps_va = pls.transform(snv_va); ps_te = pls.transform(snv_te)
        pp_aug = pls.predict(snv_aug).ravel().reshape(-1,1)
        pp_va = pls.predict(snv_va).ravel().reshape(-1,1)
        pp_te = pls.predict(snv_te).ravel().reshape(-1,1)
        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean'); knn.fit(ps_orig)
        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1,1)
        _, iv = knn.kneighbors(ps_va,5); knn_va = np.mean(y_tr[iv],axis=1).reshape(-1,1)
        _, it_ = knn.kneighbors(ps_te,5); knn_te = np.mean(y_tr[it_],axis=1).reshape(-1,1)
        r_aug = (X_aug[:,idx_1940]/(X_aug[:,idx_1300]+1e-8)).reshape(-1,1)
        r_va = (X_va[:,idx_1940]/(X_va[:,idx_1300]+1e-8)).reshape(-1,1)
        r_te = (X_test_raw[:,idx_1940]/(X_test_raw[:,idx_1300]+1e-8)).reshape(-1,1)
        s_aug = np.std(X_aug,axis=1,keepdims=True)
        s_va = np.std(X_va,axis=1,keepdims=True)
        s_te = np.std(X_test_raw,axis=1,keepdims=True)
        # 散乱帯密度プロキシ追加
        sc_aug = np.mean(X_aug[:,scatter_band],axis=1,keepdims=True)
        sc_va = np.mean(X_va[:,scatter_band],axis=1,keepdims=True)
        sc_te = np.mean(X_test_raw[:,scatter_band],axis=1,keepdims=True)
        ft = np.hstack([ps_aug,pp_aug,knn_aug,r_aug,s_aug,sc_aug])
        fv = np.hstack([ps_va,pp_va,knn_va,r_va,s_va,sc_va])
        fe = np.hstack([ps_te,pp_te,knn_te,r_te,s_te,sc_te])
        model = lgb.LGBMRegressor(n_estimators=1000,learning_rate=0.03,max_depth=4,
            num_leaves=15,subsample=0.8,colsample_bytree=0.8,random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(30,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  ImpPLS{n_comp} seed={seed} OOF={oof_rmse:.4f}")
    return {'name':f'ImpPLS{n_comp}_s{seed}','oof':oof_rmse,'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# モデルC: Two-Stage 改善版（scatter_band密度補正）
# ============================================================
def run_two_stage_improved(lgb_seed=7, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, 300, 200, 0.3, lgb_seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_orig = apply_snv(X_tr); snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va); snv_te = apply_snv(X_test_raw)
        d1_orig = savgol_filter(snv_orig,15,2,deriv=1,axis=1)
        d1_aug = savgol_filter(snv_aug,15,2,deriv=1,axis=1)
        d1_va = savgol_filter(snv_va,15,2,deriv=1,axis=1)
        d1_te = savgol_filter(snv_te,15,2,deriv=1,axis=1)
        pls2 = PLSRegression(n_components=2, scale=False); pls2.fit(snv_orig, y_tr)
        pls2_pred_aug=pls2.predict(snv_aug).ravel(); pls2_pred_va=pls2.predict(snv_va).ravel()
        pls2_pred_te=pls2.predict(snv_te).ravel()
        pls2_score_aug=pls2.transform(snv_aug); pls2_score_va=pls2.transform(snv_va)
        pls2_score_te=pls2.transform(snv_te)
        pls5 = PLSRegression(n_components=5, scale=False); pls5.fit(d1_orig, y_tr)
        pls5_pred_aug=pls5.predict(d1_aug).ravel(); pls5_pred_va=pls5.predict(d1_va).ravel()
        pls5_pred_te=pls5.predict(d1_te).ravel()
        scaler_r = StandardScaler(); snv_orig_s = scaler_r.fit_transform(snv_orig)
        ridge = Ridge(alpha=100.0); ridge.fit(snv_orig_s, y_tr)
        ridge_pred_aug=ridge.predict(scaler_r.transform(snv_aug))
        ridge_pred_va=ridge.predict(scaler_r.transform(snv_va))
        ridge_pred_te=ridge.predict(scaler_r.transform(snv_te))
        pls2_score_orig = pls2.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean'); knn.fit(pls2_score_orig)
        _, ik = knn.kneighbors(pls2_score_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv = knn.kneighbors(pls2_score_va,5); knn_va = np.mean(y_tr[iv],axis=1)
        _, it = knn.kneighbors(pls2_score_te,5); knn_te = np.mean(y_tr[it],axis=1)
        pca = PCA(n_components=10, random_state=42); pca.fit(snv_orig)
        pc_aug=pca.transform(snv_aug); pc_va=pca.transform(snv_va); pc_te=pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn2 = NearestNeighbors(n_neighbors=5, metric='cosine'); knn2.fit(pc_orig)
        _, ik2 = knn2.kneighbors(pc_aug, n_neighbors=6)
        knn2_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik2[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn2_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv2 = knn2.kneighbors(pc_va,5); knn2_va = np.mean(y_tr[iv2],axis=1)
        _, it2 = knn2.kneighbors(pc_te,5); knn2_te = np.mean(y_tr[it2],axis=1)
        phys_aug, _ = extract_physics_features(X_aug)
        phys_va, _ = extract_physics_features(X_va)
        phys_te, _ = extract_physics_features(X_test_raw)
        ft = np.column_stack([pls2_pred_aug,pls5_pred_aug,ridge_pred_aug,
            knn_aug,knn2_aug,pls2_score_aug,pc_aug,phys_aug])
        fv = np.column_stack([pls2_pred_va,pls5_pred_va,ridge_pred_va,
            knn_va,knn2_va,pls2_score_va,pc_va,phys_va])
        fe = np.column_stack([pls2_pred_te,pls5_pred_te,ridge_pred_te,
            knn_te,knn2_te,pls2_score_te,pc_te,phys_te])
        model = lgb.LGBMRegressor(n_estimators=1500,learning_rate=0.02,max_depth=4,
            num_leaves=15,subsample=0.8,colsample_bytree=0.7,reg_alpha=0.1,reg_lambda=1.0,
            min_child_samples=20,random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(50,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  ImpTS seed={lgb_seed} OOF={oof_rmse:.4f}")
    return {'name':f'ImpTS_s{lgb_seed}','oof':oof_rmse,
            'test_pred':final_pred,'oof_pred':oof_pred}


# ============================================================
# 実行
# ============================================================
print("=" * 60)
print("🚀 LB=11.198を起点: 全モデル改善版 + 精密比率探索")
print("=" * 60)

results = {}

# 全モデル改善版を生成
print("\n📌 改善版モデル全面展開")

# A: 改善版Base（複数seed）
for s in [35, 42]:
    results[f'ib_{s}'] = run_lgb_improved(seed=s)

# B: 改善版PLS
for s in [42, 0]:
    results[f'ip_{s}'] = run_pls_improved(n_comp=2, seed=s)

# C: 改善版TS（複数seed）
for s in [7, 33, 35]:
    results[f'it_{s}'] = run_two_stage_improved(lgb_seed=s)

# 旧版も参照用に
print("\n📌 旧版（参照用）")
from sklearn.cross_decomposition import PLSRegression as PLS_

def run_pls_old(n_comp=2, seed=42, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test)); oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed+fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_orig = apply_snv(X_tr); snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va); snv_te = apply_snv(X_test_raw)
        pls = PLSRegression(n_components=n_comp, scale=False); pls.fit(snv_orig, y_tr)
        ps_aug=pls.transform(snv_aug); ps_va=pls.transform(snv_va); ps_te=pls.transform(snv_te)
        pp_aug=pls.predict(snv_aug).ravel().reshape(-1,1)
        pp_va=pls.predict(snv_va).ravel().reshape(-1,1)
        pp_te=pls.predict(snv_te).ravel().reshape(-1,1)
        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean'); knn.fit(ps_orig)
        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb!=i][:5] if i<n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1,1)
        _, iv = knn.kneighbors(ps_va,5); knn_va = np.mean(y_tr[iv],axis=1).reshape(-1,1)
        _, it_ = knn.kneighbors(ps_te,5); knn_te = np.mean(y_tr[it_],axis=1).reshape(-1,1)
        r_aug=(X_aug[:,idx_1940]/(X_aug[:,idx_1300]+1e-8)).reshape(-1,1)
        r_va=(X_va[:,idx_1940]/(X_va[:,idx_1300]+1e-8)).reshape(-1,1)
        r_te=(X_test_raw[:,idx_1940]/(X_test_raw[:,idx_1300]+1e-8)).reshape(-1,1)
        s_aug=np.std(X_aug,axis=1,keepdims=True); s_va=np.std(X_va,axis=1,keepdims=True)
        s_te=np.std(X_test_raw,axis=1,keepdims=True)
        ft=np.hstack([ps_aug,pp_aug,knn_aug,r_aug,s_aug])
        fv=np.hstack([ps_va,pp_va,knn_va,r_va,s_va])
        fe=np.hstack([ps_te,pp_te,knn_te,r_te,s_te])
        model = lgb.LGBMRegressor(n_estimators=1000,learning_rate=0.03,max_depth=4,
            num_leaves=15,subsample=0.8,colsample_bytree=0.8,random_state=42,verbosity=-1)
        model.fit(ft,y_aug,eval_set=[(fv,y_va)],callbacks=[lgb.early_stopping(30,verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe))/5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  OldPLS{n_comp} seed={seed} OOF={oof_rmse:.4f}")
    return {'name':f'OldPLS{n_comp}_s{seed}','oof':oof_rmse,'test_pred':final_pred,'oof_pred':oof_pred}

results['op_42'] = run_pls_old(n_comp=2, seed=42)


# ============================================================
# 精密ブレンド探索（0.025刻み）
# ============================================================
print(f"\n{'='*60}")
print("🔬 精密ブレンド探索")
print(f"{'='*60}")

blend_results = []

base_keys = [k for k in results if k.startswith('ib_')]
ts_keys = [k for k in results if k.startswith('it_')]
pls_keys = [k for k in results if k.startswith('ip_') or k.startswith('op_')]

for bk in base_keys:
    for tk in ts_keys:
        for pk in pls_keys:
            for wb in np.arange(0.425, 0.675, 0.025):
                for wt in np.arange(0.125, 0.325, 0.025):
                    wp = round(1.0 - wb - wt, 3)
                    if wp < 0.10 or wp > 0.40:
                        continue
                    oof_bl = (wb * results[bk]['oof_pred'] +
                              wt * results[tk]['oof_pred'] +
                              wp * results[pk]['oof_pred'])
                    rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
                    pred_bl = (wb * results[bk]['test_pred'] +
                               wt * results[tk]['test_pred'] +
                               wp * results[pk]['test_pred'])
                    blend_results.append({
                        'name': f"{bk}x{wb:.3f}+{tk}x{wt:.3f}+{pk}x{wp:.3f}",
                        'bk': bk, 'tk': tk, 'pk': pk,
                        'wb': wb, 'wt': wt, 'wp': wp,
                        'oof': rmse_bl, 'test_pred': pred_bl,
                    })

blend_results.sort(key=lambda x: x['oof'])

print(f"\n  Top 20:")
print(f"  {'Name':<60s} {'OOF':>8s}")
print(f"  {'─'*60} {'─'*8}")
for b in blend_results[:20]:
    print(f"  {b['name']:<60s} {b['oof']:>8.4f}")

# wb別最良の表示
print(f"\n  Base比率別ベストOOF:")
for wb_t in [0.45, 0.50, 0.525, 0.55, 0.575, 0.60, 0.625]:
    cands = [b for b in blend_results if abs(b['wb']-wb_t)<0.015]
    if cands:
        best = min(cands, key=lambda x: x['oof'])
        print(f"    wb≈{wb_t:.3f}: OOF={best['oof']:.4f} ({best['tk']}, {best['pk']})")

# 改善版PLS vs 旧版PLS
print(f"\n  PLS改善版 vs 旧版 (同比率比較):")
for wb_t in [0.55, 0.575, 0.60]:
    imp_best = None; old_best = None
    for b in blend_results:
        if abs(b['wb']-wb_t)<0.015 and b['bk']=='ib_35' and b['tk']=='it_7':
            if 'ip_' in b['pk'] and imp_best is None: imp_best = b
            if 'op_' in b['pk'] and old_best is None: old_best = b
    if imp_best and old_best:
        print(f"    wb≈{wb_t:.2f}: 改善PLS={imp_best['oof']:.4f} 旧PLS={old_best['oof']:.4f} "
              f"差={imp_best['oof']-old_best['oof']:.4f}")


# ============================================================
# 提出ファイル（厳選3個）
# ============================================================
print(f"\n{'='*60}")
print("📁 提出ファイル作成（厳選3個）")
print(f"{'='*60}")

submissions = {}

# 1. 全改善版3Mの最良（OOFベスト付近でwb=0.55前後）
best_all_imp = None
for b in blend_results:
    if 'ip_' in b['pk'] and 'ib_' in b['bk'] and 'it_' in b['tk']:
        best_all_imp = b
        break
if best_all_imp:
    out = submit_template.copy()
    out[1] = np.clip(best_all_imp['test_pred'], 0, None)
    fname = "submission_all_improved.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_all_imp['oof']
    print(f"  ✅ {fname} (OOF={best_all_imp['oof']:.4f})")
    print(f"     [{best_all_imp['name']}]")

# 2. 改善Base+改善TS + 旧PLS（LB=11.198と同系統で比率微調整）
best_oldpls = None
for b in blend_results:
    if 'op_' in b['pk'] and 'ib_' in b['bk'] and 'it_' in b['tk']:
        best_oldpls = b
        break
if best_oldpls:
    out = submit_template.copy()
    out[1] = np.clip(best_oldpls['test_pred'], 0, None)
    fname = "submission_imp_oldpls.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_oldpls['oof']
    print(f"  ✅ {fname} (OOF={best_oldpls['oof']:.4f})")
    print(f"     [{best_oldpls['name']}]")

# 3. 異なるTS seedの構成（多様性）
best_diff = None
used_ts = set()
if best_all_imp: used_ts.add(best_all_imp['tk'])
if best_oldpls: used_ts.add(best_oldpls['tk'])
for b in blend_results:
    if b['tk'] not in used_ts:
        best_diff = b
        break
if best_diff:
    out = submit_template.copy()
    out[1] = np.clip(best_diff['test_pred'], 0, None)
    fname = "submission_diverse_ts.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_diff['oof']
    print(f"  ✅ {fname} (OOF={best_diff['oof']:.4f})")
    print(f"     [{best_diff['name']}]")


# ============================================================
# 最終分析
# ============================================================
print(f"\n{'='*60}")
print("📌 最終分析")
print(f"{'='*60}")

print(f"\n  単独モデルOOF:")
for k in sorted(results, key=lambda k: results[k]['oof']):
    r = results[k]
    tag = " [LB=11.198系]" if 'ib_' in k or 'it_' in k else ""
    print(f"    {r['name']:<25s} OOF={r['oof']:.4f}{tag}")

print(f"\n  LBスコア履歴:")
print(f"    旧3M (b0.60+t0.20+p0.20)      → LB=11.330")
print(f"    改善版ブレンド                  → LB=11.198 ★BEST")
print(f"    改善の鍵: scatter_band密度補正")

print(f"\n  提出候補:")
for fname, oof in sorted(submissions.items(), key=lambda x: x[1]):
    short = fname.replace("submission_","").replace(".csv","")
    print(f"    {short:<30s} OOF={oof:.4f}")

print(f"""
■ 戦略:
  LB=11.198の鍵は scatter_band(1100-1250nm) 密度補正。
  今回は:
  1. PLS-onlyにも同じ密度補正を追加 → 3モデル全部改善版
  2. 比率の精密調整 (0.025刻み)
  3. TS seed多様性の確保

■ 推奨提出順:
  1st: all_improved (全モデル改善版)
  2nd: imp_oldpls (改善Base+TS + 旧PLS → LB=11.198系統の比率微調整)
  3rd: diverse_ts (異なるTS seedで多様性)
""")

# Submission_trap_overall_best.csv

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 0. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit_template = pd.read_csv('data/sample_submit.csv', header=None)
train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
y_true = np.expm1(y_train_log)
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)

# 波長インデックス
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1450 = np.argmin(np.abs(wavelengths - 1450))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))
idx_1700 = np.argmin(np.abs(wavelengths - 1700))
idx_2330 = np.argmin(np.abs(wavelengths - 2330))

# バンド定義
scatter_band  = (wavelengths >= 1100) & (wavelengths <= 1250)
water_band1   = (wavelengths >= 1350) & (wavelengths <= 1600)   # O-H第1倍音
water_band2   = (wavelengths >= 1800) & (wavelengths <= 2100)   # O-H結合音
water_weak    = (wavelengths >= 1400) & (wavelengths <= 1520)   # 弱い水バンド（飽和しにくい）
water_strong  = (wavelengths >= 1880) & (wavelengths <= 1980)   # 強い水バンド（飽和しやすい）
cellulose_band = (wavelengths >= 2050) & (wavelengths <= 2200)  # セルロース関連

X_train_raw = train[spec_cols].values
X_test_raw  = test[spec_cols].values


# ============================================================
# 1. データ探索：罠の仮説検証
# ============================================================
print("=" * 70)
print("📊 PART 1: データ探索 ─ 罠の仮説検証")
print("=" * 70)

mc = train['含水率']

# Q1: 含水率の分布とFSP付近のサンプル数
print("\n■ Q1: 含水率分布")
print(f"  min={mc.min():.1f}, max={mc.max():.1f}, mean={mc.mean():.1f}, median={mc.median():.1f}")
bins = [0, 10, 20, 30, 50, 100, 150, 300]
for i in range(len(bins)-1):
    n = ((mc >= bins[i]) & (mc < bins[i+1])).sum()
    print(f"  [{bins[i]:>3d}-{bins[i+1]:>3d}%): {n:>4d} samples ({n/len(mc)*100:>5.1f}%)")

# Q2: 訓練 vs テストの樹種
print("\n■ Q2: 樹種の分布")
train_species = sorted(train['樹種'].unique())
test_species  = sorted(test['樹種'].unique())
overlap = set(train_species) & set(test_species)
print(f"  訓練: {len(train_species)}種 → {train_species}")
print(f"  テスト: {len(test_species)}種 → {test_species}")
print(f"  重複: {len(overlap)}種 → {overlap if overlap else 'なし（完全ドメインシフト）'}")

print("\n  樹種別サンプル数とMC範囲（訓練）:")
for sp, grp in train.groupby('樹種'):
    print(f"    {sp:<12s}: n={len(grp):>3d}, MC=[{grp['含水率'].min():>6.1f}, {grp['含水率'].max():>6.1f}]")

# Q3: 散乱帯は密度のプロキシとして機能するか？（罠5, 10検証）
print("\n■ Q3: 散乱帯 vs 含水率（罠10検証）")
scatter_vals = np.mean(X_train_raw[:, scatter_band], axis=1)
for sp, grp in train.groupby('樹種'):
    sp_idx = grp.index
    sc = scatter_vals[sp_idx]
    mc_sp = grp['含水率']
    # 低MC時と高MC時の散乱帯を比較
    low  = mc_sp < 30
    high = mc_sp > 80
    if low.sum() > 0 and high.sum() > 0:
        sc_low  = sc[low.values].mean()
        sc_high = sc[high.values].mean()
        print(f"    {sp:<12s}: scatter低MC={sc_low:.4f}, scatter高MC={sc_high:.4f}, "
              f"差={sc_high-sc_low:.4f} ({'含水率で変動→罠10確認' if abs(sc_high-sc_low)>0.02 else '安定'})")

# Q4: ピークシフトの実態（罠3, 7検証）
print("\n■ Q4: 水バンド2のピーク位置シフト（罠3検証）")
water2_wl = wavelengths[water_band2]
water2_region = X_train_raw[:, water_band2]
peak_wl = water2_wl[np.argmax(water2_region, axis=1)]

for mc_range, label in [((0, 20), "0-20%"), ((20, 35), "20-35%(FSP付近)"),
                         ((50, 100), "50-100%"), ((100, 999), ">100%")]:
    mask = (mc >= mc_range[0]) & (mc < mc_range[1])
    if mask.sum() > 0:
        print(f"    MC {label:<20s}: peak={peak_wl[mask].mean():.1f}±{peak_wl[mask].std():.1f} nm "
              f"(n={mask.sum()})")

# Q5: 吸光度の飽和確認（罠8検証）
print("\n■ Q5: 1940nm吸光度の飽和（罠8検証）")
abs_1940 = X_train_raw[:, idx_1940]
abs_1450 = X_train_raw[:, idx_1450]
for mc_range, label in [((0, 30), "0-30%"), ((30, 80), "30-80%"),
                         ((80, 150), "80-150%"), ((150, 999), ">150%")]:
    mask = (mc >= mc_range[0]) & (mc < mc_range[1])
    if mask.sum() > 0:
        print(f"    MC {label:<10s}: Abs1940={abs_1940[mask].mean():.4f}, "
              f"Abs1450={abs_1450[mask].mean():.4f}, "
              f"ratio1450/1940={abs_1450[mask].mean()/abs_1940[mask].mean():.4f}")


# ============================================================
# 2. 前処理関数
# ============================================================
def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

def mixup_cross_species(X, y, species, n_augment=500, alpha=0.3, seed=42):
    rng = np.random.RandomState(seed)
    unique_species = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_species, size=2, replace=False)
        idx1 = rng.choice(np.where(species == sp1)[0])
        idx2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[idx1] + (1 - lam) * X[idx2])
        y_aug.append(lam * y[idx1] + (1 - lam) * y[idx2])
    return np.array(X_aug), np.array(y_aug)

def mixup_hybrid(X, y, species, n_cross=300, n_within=200, alpha=0.3, seed=42):
    X_c, y_c = mixup_cross_species(X, y, species, n_cross, alpha, seed)
    rng = np.random.RandomState(seed + 1000)
    unique_species = np.unique(species)
    X_w, y_w = [], []
    for _ in range(n_within):
        sp = rng.choice(unique_species)
        sp_idx = np.where(species == sp)[0]
        if len(sp_idx) < 2:
            continue
        i1, i2 = rng.choice(sp_idx, size=2, replace=False)
        lam = rng.beta(alpha, alpha)
        X_w.append(lam * X[i1] + (1 - lam) * X[i2])
        y_w.append(lam * y[i1] + (1 - lam) * y[i2])
    X_w, y_w = np.array(X_w), np.array(y_w)
    return np.vstack([X_c, X_w]), np.concatenate([y_c, y_w])


# ============================================================
# 3. 改善版物理特徴量（罠1,3,7,8,9,10対策）
# ============================================================
def extract_trap_features(X_raw):
    """
    各罠に対応した物理的に意味のある特徴量。
    設計原則：「樹種の指紋」ではなく「水の物理状態」を捉える。
    """
    feats = {}

    # --- 各バンドの基本統計量（raw） ---
    sc_mean  = np.mean(X_raw[:, scatter_band], axis=1)
    w1_mean  = np.mean(X_raw[:, water_band1], axis=1)
    w2_mean  = np.mean(X_raw[:, water_band2], axis=1)
    ww_mean  = np.mean(X_raw[:, water_weak], axis=1)
    ws_mean  = np.mean(X_raw[:, water_strong], axis=1)
    cel_mean = np.mean(X_raw[:, cellulose_band], axis=1)

    # ===== 罠9: 密度×水分の乗法的交互作用 =====
    # 核心: 水バンドの強度を散乱帯で割る = 密度を正規化
    feats['w2_div_scatter'] = w2_mean / (sc_mean + 1e-8)
    feats['ws_div_scatter'] = ws_mean / (sc_mean + 1e-8)
    feats['ww_div_scatter'] = ww_mean / (sc_mean + 1e-8)
    feats['cel_div_scatter'] = cel_mean / (sc_mean + 1e-8)

    # 水バンド同士の比率（密度の影響がキャンセルされる）
    feats['w2_div_w1'] = w2_mean / (w1_mean + 1e-8)
    feats['ws_div_ww'] = ws_mean / (ww_mean + 1e-8)  # 強い帯/弱い帯 → 飽和度指標

    # ===== 罠3,7: ピークシフト・FSP不連続 =====
    water2_region = X_raw[:, water_band2]
    water2_wl_local = wavelengths[water_band2]
    peak_idx_local = np.argmax(water2_region, axis=1)
    feats['peak_wl_w2'] = water2_wl_local[peak_idx_local]  # ピーク位置

    # ピークの非対称性（左右の面積比）→ 自由水/結合水の混在度
    peak_abs = np.max(water2_region, axis=1)
    half_max = peak_abs / 2.0
    # 半値幅の左右端を近似
    left_area = np.zeros(len(X_raw))
    right_area = np.zeros(len(X_raw))
    for i in range(len(X_raw)):
        pi = peak_idx_local[i]
        left_area[i] = np.sum(water2_region[i, :pi+1])
        right_area[i] = np.sum(water2_region[i, pi:])
    feats['peak_asymmetry'] = left_area / (right_area + 1e-8)

    # ===== 罠8: 高MC域の吸光度飽和 =====
    # 飽和しにくい1450nm / 飽和しやすい1940nmの比率 → 飽和度指標
    feats['abs1450_div_1940'] = X_raw[:, idx_1450] / (X_raw[:, idx_1940] + 1e-8)
    # 弱バンド/強バンド比率（飽和が進むほど1に近づく）
    feats['weak_div_strong'] = ww_mean / (ws_mean + 1e-8)

    # ===== 罠1: 木材O-Hとの分離 =====
    # セルロースO-H帯と水帯の比率 → 木質O-Hの寄与推定
    feats['cel_div_w2'] = cel_mean / (w2_mean + 1e-8)
    feats['cel_div_w1'] = cel_mean / (w1_mean + 1e-8)

    # ===== 罠5,10: 密度プロキシ（raw情報の保持） =====
    feats['scatter_mean'] = sc_mean
    feats['scatter_std'] = np.std(X_raw[:, scatter_band], axis=1)
    feats['raw_mean'] = np.mean(X_raw, axis=1)
    feats['raw_std'] = np.std(X_raw, axis=1)

    # ===== 罠2: 散乱補正の残差指標 =====
    # SNV後の残差パターンの指標
    snv = apply_snv(X_raw)
    feats['snv_scatter_mean'] = np.mean(snv[:, scatter_band], axis=1)  # SNV後も散乱帯に残る情報

    # ===== 罠7対策: FSP境界指標 =====
    # 1次微分のwater2帯の特徴 → FSP前後で形状が変化
    d1 = savgol_filter(snv, 15, 2, deriv=1, axis=1)
    feats['d1_w2_max'] = np.max(d1[:, water_band2], axis=1)
    feats['d1_w2_min'] = np.min(d1[:, water_band2], axis=1)
    feats['d1_w2_range'] = feats['d1_w2_max'] - feats['d1_w2_min']

    # 2次微分のピーク形状（水ピークの曲率 → 飽和・シフトの指標）
    d2 = savgol_filter(snv, 15, 2, deriv=2, axis=1)
    feats['d2_at1940'] = d2[:, idx_1940]
    feats['d2_at1450'] = d2[:, idx_1450]

    return pd.DataFrame(feats).values, list(feats.keys())


# ============================================================
# 4. モデルA改善: ベースLGB（全罠対策入り）
# ============================================================
def run_lgb_trap_aware(seed=35, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
            X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values

        # Mixup
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        # SNV + 微分
        snv_aug = apply_snv(X_aug)
        d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        snv_va = apply_snv(X_va)
        d1_va = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        snv_te = apply_snv(X_test_raw)
        d1_te = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        # PCA（SNV後のオリジナルデータのみでfit）
        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va = pca.transform(snv_va)
        pc_te = pca.transform(snv_te)

        # KNN
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pc_orig)

        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)

        _, iv = knn.kneighbors(pc_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it = knn.kneighbors(pc_te, 5)
        knn_te = np.mean(y_tr[it], axis=1).reshape(-1, 1)

        # 罠対策特徴量
        trap_aug, trap_names = extract_trap_features(X_aug)
        trap_va, _ = extract_trap_features(X_va)
        trap_te, _ = extract_trap_features(X_test_raw)

        # 特徴量結合
        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, trap_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, trap_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, trap_te])

        model = lgb.LGBMRegressor(
            n_estimators=1500, learning_rate=0.025, max_depth=5,
            num_leaves=31, subsample=0.8, colsample_bytree=0.25,
            min_child_samples=15, reg_alpha=0.1, reg_lambda=1.0,
            random_state=42, verbosity=-1
        )
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe)) / 5

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  ModelA(TrapLGB) seed={seed} OOF={oof_rmse:.4f}")
    return {'name': f'TrapLGB_s{seed}', 'oof': oof_rmse,
            'test_pred': final_pred, 'oof_pred': oof_pred}


# ============================================================
# 5. モデルB改善: PLS-LGB（罠9の交互作用を重視）
# ============================================================
def run_pls_trap_aware(n_comp=2, seed=42, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
            X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_orig = apply_snv(X_tr)
        snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va)
        snv_te = apply_snv(X_test_raw)

        # PLS fit on original only
        pls = PLSRegression(n_components=n_comp, scale=False)
        pls.fit(snv_orig, y_tr)
        ps_aug = pls.transform(snv_aug)
        ps_va = pls.transform(snv_va)
        ps_te = pls.transform(snv_te)
        pp_aug = pls.predict(snv_aug).ravel().reshape(-1, 1)
        pp_va = pls.predict(snv_va).ravel().reshape(-1, 1)
        pp_te = pls.predict(snv_te).ravel().reshape(-1, 1)

        # KNN in PLS space
        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(ps_orig)
        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)
        _, iv = knn.kneighbors(ps_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it_ = knn.kneighbors(ps_te, 5)
        knn_te = np.mean(y_tr[it_], axis=1).reshape(-1, 1)

        # 罠対策特徴量（PLS版は絞る：交互作用と密度のみ）
        trap_aug, _ = extract_trap_features(X_aug)
        trap_va, _  = extract_trap_features(X_va)
        trap_te, _  = extract_trap_features(X_test_raw)

        # PLS版は特徴量を抑制（過学習防止）
        # trap_features の中から重要度の高い列のみ使用
        # → 全部入れるとPLSの圧縮メリットが消えるので、交互作用系のみ
        # indices: w2_div_scatter(0), w1_div_scatter(1), ws_div_scatter(2),
        #          ww_div_scatter(3), w2_div_w1(5), ws_div_ww(6),
        #          scatter_mean(15), raw_std(18)
        selected_trap = [0, 1, 2, 3, 5, 6, 7, 8, 9, 15, 18]
        trap_aug_sel = trap_aug[:, selected_trap]
        trap_va_sel = trap_va[:, selected_trap]
        trap_te_sel = trap_te[:, selected_trap]

        ft = np.hstack([ps_aug, pp_aug, knn_aug, trap_aug_sel])
        fv = np.hstack([ps_va, pp_va, knn_va, trap_va_sel])
        fe = np.hstack([ps_te, pp_te, knn_te, trap_te_sel])

        model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03, max_depth=4,
            num_leaves=15, subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=-1
        )
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe)) / 5

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  ModelB(TrapPLS{n_comp}) seed={seed} OOF={oof_rmse:.4f}")
    return {'name': f'TrapPLS{n_comp}_s{seed}', 'oof': oof_rmse,
            'test_pred': final_pred, 'oof_pred': oof_pred}


# ============================================================
# 6. モデルC改善: Two-Stage（罠対策フル装備）
# ============================================================
def run_two_stage_trap_aware(lgb_seed=7, verbose=True):
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
            X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values

        X_mix, y_mix = mixup_hybrid(X_tr, y_tr, tr_sp, 300, 200, 0.3, lgb_seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])

        snv_orig = apply_snv(X_tr)
        snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va)
        snv_te = apply_snv(X_test_raw)

        d1_orig = savgol_filter(snv_orig, 15, 2, deriv=1, axis=1)
        d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        d1_va = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        d1_te = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)

        # Stage1: 線形モデル予測値
        pls2 = PLSRegression(n_components=2, scale=False)
        pls2.fit(snv_orig, y_tr)
        pls2_pred_aug = pls2.predict(snv_aug).ravel()
        pls2_pred_va = pls2.predict(snv_va).ravel()
        pls2_pred_te = pls2.predict(snv_te).ravel()
        pls2_score_aug = pls2.transform(snv_aug)
        pls2_score_va = pls2.transform(snv_va)
        pls2_score_te = pls2.transform(snv_te)

        pls5 = PLSRegression(n_components=5, scale=False)
        pls5.fit(d1_orig, y_tr)
        pls5_pred_aug = pls5.predict(d1_aug).ravel()
        pls5_pred_va = pls5.predict(d1_va).ravel()
        pls5_pred_te = pls5.predict(d1_te).ravel()

        scaler_r = StandardScaler()
        snv_orig_s = scaler_r.fit_transform(snv_orig)
        ridge = Ridge(alpha=100.0)
        ridge.fit(snv_orig_s, y_tr)
        ridge_pred_aug = ridge.predict(scaler_r.transform(snv_aug))
        ridge_pred_va = ridge.predict(scaler_r.transform(snv_va))
        ridge_pred_te = ridge.predict(scaler_r.transform(snv_te))

        # KNN in PLS space
        pls2_score_orig = pls2.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
        knn.fit(pls2_score_orig)
        _, ik = knn.kneighbors(pls2_score_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv = knn.kneighbors(pls2_score_va, 5)
        knn_va = np.mean(y_tr[iv], axis=1)
        _, it = knn.kneighbors(pls2_score_te, 5)
        knn_te = np.mean(y_tr[it], axis=1)

        # KNN in PCA space
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug)
        pc_va = pca.transform(snv_va)
        pc_te = pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn2 = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn2.fit(pc_orig)
        _, ik2 = knn2.kneighbors(pc_aug, n_neighbors=6)
        knn2_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik2[i]
            v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn2_aug[i] = np.mean(y_tr[:n_orig][v])
        _, iv2 = knn2.kneighbors(pc_va, 5)
        knn2_va = np.mean(y_tr[iv2], axis=1)
        _, it2 = knn2.kneighbors(pc_te, 5)
        knn2_te = np.mean(y_tr[it2], axis=1)

        # 罠対策特徴量
        trap_aug, _ = extract_trap_features(X_aug)
        trap_va, _ = extract_trap_features(X_va)
        trap_te, _ = extract_trap_features(X_test_raw)

        ft = np.column_stack([pls2_pred_aug, pls5_pred_aug, ridge_pred_aug,
                              knn_aug, knn2_aug, pls2_score_aug, pc_aug, trap_aug])
        fv = np.column_stack([pls2_pred_va, pls5_pred_va, ridge_pred_va,
                              knn_va, knn2_va, pls2_score_va, pc_va, trap_va])
        fe = np.column_stack([pls2_pred_te, pls5_pred_te, ridge_pred_te,
                              knn_te, knn2_te, pls2_score_te, pc_te, trap_te])

        model = lgb.LGBMRegressor(
            n_estimators=1500, learning_rate=0.02, max_depth=4,
            num_leaves=15, subsample=0.8, colsample_bytree=0.6,
            reg_alpha=0.1, reg_lambda=1.0,
            min_child_samples=20, random_state=42, verbosity=-1
        )
        model.fit(ft, y_aug,
                  eval_set=[(fv, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe)) / 5

    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose:
        print(f"  ModelC(TrapTS) seed={lgb_seed} OOF={oof_rmse:.4f}")
    return {'name': f'TrapTS_s{lgb_seed}', 'oof': oof_rmse,
            'test_pred': final_pred, 'oof_pred': oof_pred}


# ============================================================
# 7. 旧モデル（比較用）
# ============================================================
def run_lgb_original(seed=35, verbose=True):
    """旧ベースLGB（LB=11.134のベースモデルA）"""
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test))
    oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
            X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix])
        y_aug = np.concatenate([y_tr, y_mix])
        snv_aug = apply_snv(X_aug); d1_aug = savgol_filter(snv_aug, 15, 2, deriv=1, axis=1)
        snv_va = apply_snv(X_va); d1_va = savgol_filter(snv_va, 15, 2, deriv=1, axis=1)
        snv_te = apply_snv(X_test_raw); d1_te = savgol_filter(snv_te, 15, 2, deriv=1, axis=1)
        r_aug = (X_aug[:, idx_1940] / (X_aug[:, idx_1300] + 1e-8)).reshape(-1, 1)
        r_va = (X_va[:, idx_1940] / (X_va[:, idx_1300] + 1e-8)).reshape(-1, 1)
        r_te = (X_test_raw[:, idx_1940] / (X_test_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
        sc_aug = np.mean(X_aug[:, scatter_band], axis=1, keepdims=True)
        sc_va = np.mean(X_va[:, scatter_band], axis=1, keepdims=True)
        sc_te = np.mean(X_test_raw[:, scatter_band], axis=1, keepdims=True)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va = np.std(X_va, axis=1, keepdims=True)
        s_te = np.std(X_test_raw, axis=1, keepdims=True)
        snv_orig = apply_snv(X_tr)
        pca = PCA(n_components=10, random_state=42); pca.fit(snv_orig)
        pc_aug = pca.transform(snv_aug); pc_va = pca.transform(snv_va); pc_te = pca.transform(snv_te)
        pc_orig = pca.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine'); knn.fit(pc_orig)
        _, ik = knn.kneighbors(pc_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)
        _, iv = knn.kneighbors(pc_va, 5); knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it = knn.kneighbors(pc_te, 5); knn_te = np.mean(y_tr[it], axis=1).reshape(-1, 1)
        ft = np.hstack([snv_aug, d1_aug, pc_aug, knn_aug, r_aug, s_aug, sc_aug])
        fv = np.hstack([snv_va, d1_va, pc_va, knn_va, r_va, s_va, sc_va])
        fe = np.hstack([snv_te, d1_te, pc_te, knn_te, r_te, s_te, sc_te])
        model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=5,
                                  num_leaves=31, subsample=0.8, colsample_bytree=0.3,
                                  min_child_samples=15, reg_alpha=0.05, reg_lambda=0.5,
                                  random_state=42, verbosity=-1)
        model.fit(ft, y_aug, eval_set=[(fv, y_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe)) / 5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  OldBase seed={seed} OOF={oof_rmse:.4f}")
    return {'name': f'OldBase_s{seed}', 'oof': oof_rmse,
            'test_pred': final_pred, 'oof_pred': oof_pred}


def run_pls_original(n_comp=2, seed=42, verbose=True):
    """旧PLS"""
    gkf = GroupKFold(n_splits=5)
    final_pred = np.zeros(len(test)); oof_pred = np.zeros(len(train))
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_raw, y_train_log, groups)):
        tr_sp = groups.iloc[tr_idx].values
        X_tr, y_tr = X_train_raw[tr_idx], y_train_log.iloc[tr_idx].values
        X_va, y_va = X_train_raw[va_idx], y_train_log.iloc[va_idx].values
        X_mix, y_mix = mixup_cross_species(X_tr, y_tr, tr_sp, 500, 0.3, seed + fold)
        n_orig = len(X_tr)
        X_aug = np.vstack([X_tr, X_mix]); y_aug = np.concatenate([y_tr, y_mix])
        snv_orig = apply_snv(X_tr); snv_aug = apply_snv(X_aug)
        snv_va = apply_snv(X_va); snv_te = apply_snv(X_test_raw)
        pls = PLSRegression(n_components=n_comp, scale=False); pls.fit(snv_orig, y_tr)
        ps_aug = pls.transform(snv_aug); ps_va = pls.transform(snv_va); ps_te = pls.transform(snv_te)
        pp_aug = pls.predict(snv_aug).ravel().reshape(-1, 1)
        pp_va = pls.predict(snv_va).ravel().reshape(-1, 1)
        pp_te = pls.predict(snv_te).ravel().reshape(-1, 1)
        ps_orig = pls.transform(snv_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='euclidean'); knn.fit(ps_orig)
        _, ik = knn.kneighbors(ps_aug, n_neighbors=6)
        knn_aug = np.zeros(len(X_aug))
        for i in range(len(X_aug)):
            nb = ik[i]; v = nb[nb != i][:5] if i < n_orig else nb[:5]
            knn_aug[i] = np.mean(y_tr[:n_orig][v])
        knn_aug = knn_aug.reshape(-1, 1)
        _, iv = knn.kneighbors(ps_va, 5); knn_va = np.mean(y_tr[iv], axis=1).reshape(-1, 1)
        _, it_ = knn.kneighbors(ps_te, 5); knn_te = np.mean(y_tr[it_], axis=1).reshape(-1, 1)
        r_aug = (X_aug[:, idx_1940] / (X_aug[:, idx_1300] + 1e-8)).reshape(-1, 1)
        r_va = (X_va[:, idx_1940] / (X_va[:, idx_1300] + 1e-8)).reshape(-1, 1)
        r_te = (X_test_raw[:, idx_1940] / (X_test_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
        s_aug = np.std(X_aug, axis=1, keepdims=True)
        s_va = np.std(X_va, axis=1, keepdims=True)
        s_te = np.std(X_test_raw, axis=1, keepdims=True)
        sc_aug = np.mean(X_aug[:, scatter_band], axis=1, keepdims=True)
        sc_va = np.mean(X_va[:, scatter_band], axis=1, keepdims=True)
        sc_te = np.mean(X_test_raw[:, scatter_band], axis=1, keepdims=True)
        ft = np.hstack([ps_aug, pp_aug, knn_aug, r_aug, s_aug, sc_aug])
        fv = np.hstack([ps_va, pp_va, knn_va, r_va, s_va, sc_va])
        fe = np.hstack([ps_te, pp_te, knn_te, r_te, s_te, sc_te])
        model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, max_depth=4,
                                  num_leaves=15, subsample=0.8, colsample_bytree=0.8,
                                  random_state=42, verbosity=-1)
        model.fit(ft, y_aug, eval_set=[(fv, y_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
        oof_pred[va_idx] = np.expm1(model.predict(fv))
        final_pred += np.expm1(model.predict(fe)) / 5
    oof_rmse = np.sqrt(mean_squared_error(y_true, oof_pred))
    if verbose: print(f"  OldPLS{n_comp} seed={seed} OOF={oof_rmse:.4f}")
    return {'name': f'OldPLS{n_comp}_s{seed}', 'oof': oof_rmse,
            'test_pred': final_pred, 'oof_pred': oof_pred}


# ============================================================
# 8. 実行
# ============================================================
print("\n" + "=" * 70)
print("🚀 PART 2: 改善モデル実行")
print("=" * 70)

results = {}

# --- 改善版モデル ---
print("\n📌 改善版（罠対策入り）モデル:")
for s in [35, 42]:
    results[f'ta_{s}'] = run_lgb_trap_aware(seed=s)

for s in [42, 0]:
    results[f'tb_{s}'] = run_pls_trap_aware(n_comp=2, seed=s)

for s in [7, 33, 35]:
    results[f'tc_{s}'] = run_two_stage_trap_aware(lgb_seed=s)

# --- 旧版モデル（比較用） ---
print("\n📌 旧版（比較用）:")
results['oa_35'] = run_lgb_original(seed=35)
results['ob_42'] = run_pls_original(n_comp=2, seed=42)


# ============================================================
# 9. ブレンド探索
# ============================================================
print(f"\n{'=' * 70}")
print("🔬 PART 3: ブレンド探索")
print(f"{'=' * 70}")

blend_results = []

a_keys = [k for k in results if k.startswith('ta_') or k.startswith('oa_')]
b_keys = [k for k in results if k.startswith('tb_') or k.startswith('ob_')]
c_keys = [k for k in results if k.startswith('tc_')]

for ak in a_keys:
    for ck in c_keys:
        for bk in b_keys:
            for wa in np.arange(0.35, 0.70, 0.025):
                for wc in np.arange(0.10, 0.40, 0.025):
                    wb = round(1.0 - wa - wc, 3)
                    if wb < 0.05 or wb > 0.45:
                        continue
                    oof_bl = (wa * results[ak]['oof_pred'] +
                              wc * results[ck]['oof_pred'] +
                              wb * results[bk]['oof_pred'])
                    rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
                    pred_bl = (wa * results[ak]['test_pred'] +
                               wc * results[ck]['test_pred'] +
                               wb * results[bk]['test_pred'])
                    blend_results.append({
                        'name': f"{ak}x{wa:.3f}+{ck}x{wc:.3f}+{bk}x{wb:.3f}",
                        'ak': ak, 'ck': ck, 'bk': bk,
                        'wa': wa, 'wc': wc, 'wb': wb,
                        'oof': rmse_bl, 'test_pred': pred_bl,
                    })

# 2モデルブレンドも探索（A+Bのみ, A+Cのみ）
for ak in a_keys:
    for bk in b_keys:
        for wa in np.arange(0.30, 0.80, 0.025):
            wb = round(1.0 - wa, 3)
            oof_bl = wa * results[ak]['oof_pred'] + wb * results[bk]['oof_pred']
            rmse_bl = np.sqrt(mean_squared_error(y_true, oof_bl))
            pred_bl = wa * results[ak]['test_pred'] + wb * results[bk]['test_pred']
            blend_results.append({
                'name': f"{ak}x{wa:.3f}+{bk}x{wb:.3f}",
                'ak': ak, 'ck': 'none', 'bk': bk,
                'wa': wa, 'wc': 0, 'wb': wb,
                'oof': rmse_bl, 'test_pred': pred_bl,
            })

blend_results.sort(key=lambda x: x['oof'])

print(f"\n  Top 30:")
print(f"  {'Rank':>4s} {'Name':<65s} {'OOF':>8s}")
print(f"  {'─' * 4} {'─' * 65} {'─' * 8}")
for i, b in enumerate(blend_results[:30]):
    marker = " ★" if 'ta_' in b.get('ak', '') and 'tc_' in b.get('ck', '') and 'tb_' in b.get('bk', '') else ""
    print(f"  {i + 1:>4d} {b['name']:<65s} {b['oof']:>8.4f}{marker}")

# 改善版のみ vs 旧版含むの比較
print(f"\n  改善版のみ TOP5:")
cnt = 0
for b in blend_results:
    if all(k.startswith('t') for k in [b.get('ak','x'), b.get('bk','x')]
           if k != 'none') and (b.get('ck','none') == 'none' or b['ck'].startswith('t')):
        print(f"    {b['name']:<65s} OOF={b['oof']:.4f}")
        cnt += 1
        if cnt >= 5:
            break

print(f"\n  旧版含む TOP5:")
cnt = 0
for b in blend_results:
    if any(k.startswith('o') for k in [b.get('ak',''), b.get('bk',''), b.get('ck','')]):
        print(f"    {b['name']:<65s} OOF={b['oof']:.4f}")
        cnt += 1
        if cnt >= 5:
            break


# ============================================================
# 10. 単独モデル性能一覧
# ============================================================
print(f"\n{'=' * 70}")
print("📌 PART 4: 単独モデルOOF一覧")
print(f"{'=' * 70}")
for k in sorted(results, key=lambda k: results[k]['oof']):
    r = results[k]
    tag = " [旧版]" if k.startswith('o') else " [改善版]"
    print(f"  {r['name']:<25s} OOF={r['oof']:.4f}{tag}")


# ============================================================
# 11. 提出ファイル作成
# ============================================================
print(f"\n{'=' * 70}")
print("📁 PART 5: 提出ファイル作成")
print(f"{'=' * 70}")

submissions = {}

# 1. 全改善版最良
best_all_new = None
for b in blend_results:
    if b.get('ck', 'none') != 'none':
        if all(k.startswith('t') for k in [b['ak'], b['bk'], b['ck']]):
            best_all_new = b
            break
if best_all_new:
    out = submit_template.copy()
    out[1] = np.clip(best_all_new['test_pred'], 0, None)
    fname = "submission_trap_all_new.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_all_new['oof']
    print(f"  ✅ {fname} (OOF={best_all_new['oof']:.4f})")
    print(f"     [{best_all_new['name']}]")

# 2. 全体最良（旧版含む）
best_overall = blend_results[0]
out = submit_template.copy()
out[1] = np.clip(best_overall['test_pred'], 0, None)
fname = "submission_trap_overall_best.csv"
out.to_csv(fname, index=False, header=False)
submissions[fname] = best_overall['oof']
print(f"  ✅ {fname} (OOF={best_overall['oof']:.4f})")
print(f"     [{best_overall['name']}]")

# 3. 旧版ベスト（安全策）
best_old = None
for b in blend_results:
    if any(k.startswith('o') for k in [b.get('ak',''), b.get('bk',''), b.get('ck','')]):
        best_old = b
        break
if best_old:
    out = submit_template.copy()
    out[1] = np.clip(best_old['test_pred'], 0, None)
    fname = "submission_trap_with_old.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_old['oof']
    print(f"  ✅ {fname} (OOF={best_old['oof']:.4f})")
    print(f"     [{best_old['name']}]")

# 4. 2モデルブレンド最良（シンプル構成）
best_2m = None
for b in blend_results:
    if b.get('ck', 'none') == 'none':
        best_2m = b
        break
if best_2m:
    out = submit_template.copy()
    out[1] = np.clip(best_2m['test_pred'], 0, None)
    fname = "submission_trap_2model.csv"
    out.to_csv(fname, index=False, header=False)
    submissions[fname] = best_2m['oof']
    print(f"  ✅ {fname} (OOF={best_2m['oof']:.4f})")
    print(f"     [{best_2m['name']}]")


# ============================================================
# 12. 最終サマリー
# ============================================================
print(f"\n{'=' * 70}")
print("📌 最終サマリー")
print(f"{'=' * 70}")

print(f"""
■ 今回の改善点（罠対策）:
  罠9  → 水バンド/散乱帯の比率特徴量（乗法的交互作用の明示化）
  罠3  → ピーク位置・非対称性の特徴量
  罠7  → 微分形状の変化を特徴量化（FSP境界指標）
  罠8  → 弱帯/強帯比率（飽和度指標）、2次微分曲率
  罠1  → セルロース/水帯の比率（木材O-H分離の補助）
  罠10 → SNV後の散乱帯残差情報を保持

■ 維持した成功要素:
  - Mixup（α=0.3, cross-species）
  - PLS 2成分の極限圧縮
  - LGB + PLS + Two-Stageの3モデルブレンド
  - GroupKFoldによる未知樹種シミュレーション
  - log1p目的変数変換

■ 新モデル設計思想:
  - 「罠対策特徴量」は樹種固有パターンではなく「水の物理状態」を捉える設計
  - 比率特徴量を中心にし、加法的な値は最小限に（汎化性重視）
  - colsample_bytreeをさらに絞り(0.25)、特徴量の一部しか使わない＝暗記抑制
  - PLS版は交互作用系の特徴量のみ選択（圧縮メリット維持）

■ 提出候補:
""")
for fname, oof in sorted(submissions.items(), key=lambda x: x[1]):
    short = fname.replace("submission_trap_", "").replace(".csv", "")
    print(f"  {short:<30s} OOF={oof:.4f}")

print(f"""
■ 推奨提出順:
  1st: overall_best（OOFベスト → LBとの相関を検証）
  2nd: all_new（全改善版のみ → 罠対策の効果検証）
  3rd: with_old（旧版混在 → 安全策）
  4th: 2model（シンプル構成 → 過学習リスク最小）
""")